# 01 — Data Preparation

Loads the raw scraped corpus, derives the analytical columns (`industry_reclassified`, `triadic_layers_count`, `arch_level_est`, `product_stack_breadth`, etc.), and exports the cleaned datasets used as input to the LLM coding pipeline.

**Note on the LLM pipeline (below):** the coding functions are included for transparency and reproducibility, but the execution cells are disabled (commented out). The coded results are already saved in `master_dataset_900.csv` — re-running the pipeline would call the Anthropic API and incur cost. It is not required to reproduce the analysis.

## Data Load

In [1]:
# ============================================================
# BLOCK 0 — Setup & Data Load (run this first, always)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

FILE_PATH = '/content/drive/MyDrive/Microsoft900cases/combined_microsoft_cases.csv'

# Colour palette
C_BLUE   = '#378ADD'
C_TEAL   = '#1D9E75'
C_PURPLE = '#7F77DD'
C_AMBER  = '#BA7517'
C_GRAY   = '#888780'
C_CORAL  = '#D85A30'
C_GREEN  = '#639922'

# Load
df = pd.read_csv(FILE_PATH, encoding='utf-8-sig')
df.columns = df.columns.str.strip()
df = df[df['Customer Name'].fillna('').str.strip() != ''].reset_index(drop=True)
print(f"Analytical sample after removing unavailable stories: {len(df)} stories")

# Pre-compute shared variables used across blocks
df['products_list'] = df['Products Used'].fillna('').apply(
    lambda x: [p.strip() for p in x.split('\n') if p.strip()])
df['product_stack_breadth'] = df['products_list'].apply(len)

df['industry_clean'] = df['Industry'].fillna('Unknown').apply(
    lambda x: x.split('\n')[0].strip() if x.strip() else 'Unknown')

# ── Full industry consolidation map (14 sectors) ─────────────────────────────
# Decisions documented in methods section:
#   Case 1: Social Svcs/Public Health → Health; Public Safety/Justice/Cities → Government
#   Case 2: All nonprofit labels unified into Nonprofit & IGO
#   Case 3: Commercial Other + Unknown + Other labels → Other / Unclassified (data quality flag)
INDUSTRY_MAP = {
    # Health
    'Health Provider':'Health','Healthcare':'Health','MedTech':'Health',
    'Health Pharma':'Health','Health and Personal Care':'Health','Health Payor':'Health',
    'Social Svcs and Public Health':'Health',
    'Social Svcs and Public HealthPublic and Community Health':'Health',
    # Financial Services
    'Financial Services':'Financial Services','Banking':'Financial Services',
    'Insurance':'Financial Services','Capital Markets':'Financial Services',
    # Professional Services
    'Professional and Business Services':'Professional Services',
    'Other Professional Services':'Professional Services',
    'IT Services and Business Advisory':'Professional Services',
    'IT Services':'Professional Services','Management Consulting Services':'Professional Services',
    'Accounting and Tax Services':'Professional Services','Legal Services':'Professional Services',
    'Staffing Services':'Professional Services','Managed Services':'Professional Services',
    # Retail & Consumer
    'Retailers':'Retail & Consumer','Retail and Consumer Goods':'Retail & Consumer',
    'Specialty Retail':'Retail & Consumer','Consumer Goods':'Retail & Consumer',
    'Consumer Goods Manufacturing':'Retail & Consumer','Grocery':'Retail & Consumer',
    'Food and Beverage Retailers':'Retail & Consumer','Food and Beverage':'Retail & Consumer',
    'Food Service':'Retail & Consumer','Accommodations':'Retail & Consumer',
    'Hospitality':'Retail & Consumer',
    # Manufacturing
    'Discrete Manufacturing':'Manufacturing','Industrials and Manufacturing':'Manufacturing',
    'Process Manufacturing':'Manufacturing','Industrial Equipment & Machinery':'Manufacturing',
    'Chemicals':'Manufacturing','Chemical':'Manufacturing','Semiconductor':'Manufacturing',
    'High Tech Electronics':'Manufacturing','Vehicle Suppliers':'Manufacturing',
    'Vehicle Supplier':'Manufacturing','Vehicle OEM':'Manufacturing',
    'Automotive':'Manufacturing','Automotive, Mobility, and Transportation':'Manufacturing',
    # Education
    'Higher Education':'Education','Primary and Secondary Education (K-12)':'Education',
    'Education':'Education','Libraries and Museums':'Education',
    # Government & Public Sector
    'Government':'Government & Public Sector','Central Government':'Government & Public Sector',
    'Local and Regional Government':'Government & Public Sector',
    'Central, Federal and Regional Government':'Government & Public Sector',
    'Regional Government':'Government & Public Sector',
    'Cities and Regions':'Government & Public Sector',
    'Public Safety and Justice':'Government & Public Sector',
    'Justice':'Government & Public Sector',
    'Government Operations and Infrastructure':'Government & Public Sector',
    'Government Operations and Infrastructure ISVs':'Government & Public Sector',
    'Defense Industrial Base':'Government & Public Sector',
    'Defense and Intelligence':'Government & Public Sector',
    # Telecom & Media
    'Telecommunications':'Telecom & Media','Telecommunications and Media':'Telecom & Media',
    'Telecommunications and Media/Telecommunications':'Telecom & Media',
    'Media and Entertainment':'Telecom & Media','Cable and Satellite':'Telecom & Media',
    'Broadcasters':'Telecom & Media','Film, Studio, Animation':'Telecom & Media',
    'Advertising':'Telecom & Media','Sports and Live Experiences':'Telecom & Media',
    'Gaming':'Telecom & Media',
    # Energy & Utilities
    'Energy and Resources':'Energy & Utilities','Power and Utilities':'Energy & Utilities',
    'Oil and Gas':'Energy & Utilities','Water & Sewage':'Energy & Utilities',
    'Mining':'Energy & Utilities','Environmental and Animal Welfare':'Energy & Utilities',
    'Energy and Resource Infrastructure Agencies':'Energy & Utilities',
    # Technology
    'Software, Data and Platforms':'Technology',
    'At scale software, data and platforms':'Technology',
    'Digital Native Startups and Unicorns':'Technology',
    'Biotech':'Technology','Data':'Technology',
    # Nonprofit & IGO
    'Nonprofit and IGO':'Nonprofit & IGO','NonProfit Organizations':'Nonprofit & IGO',
    'NonProfit Health and Human Srvc':'Nonprofit & IGO',
    'Nonprofit Fundraising':'Nonprofit & IGO',
    'Membership Organizations':'Nonprofit & IGO','Global Development and Aid':'Nonprofit & IGO',
    # Transport & Logistics
    'Transport and Travel':'Transport & Logistics',
    'Transport and Logistics':'Transport & Logistics','Mobility':'Transport & Logistics',
    # Real Estate & Construction
    'Real Estate':'Real Estate & Construction','Construction':'Real Estate & Construction',
    'Architecture and Engineering':'Real Estate & Construction',
    'Property Developers':'Real Estate & Construction',
    'Property Managers':'Real Estate & Construction',
    # Other / Unclassified (Case 3 — flag as data quality issue in methods)
    'Commercial Other Industries':'Other / Unclassified',
    'Unknown':'Other / Unclassified',
    'Other - Unsegmented':'Other / Unclassified',
    'Other Services':'Other / Unclassified',
}
df['industry_grouped'] = df['industry_clean'].apply(
    lambda x: INDUSTRY_MAP.get(x, 'Other / Unclassified'))
df['org_clean'] = df['Organization Size'].str.strip()

L3_KW = ['Foundry','Azure OpenAI','Azure AI','Azure Container','Azure Compute',
          'Azure Blob','Azure Cosmos','Azure Data','Azure Speech','Azure Logic',
          'Azure Function','Azure Kubernetes','Azure Machine','Azure ML']
L2_KW = ['Copilot Studio','Power Automate','Power Apps','Power Platform']

def estimate_level(products_str):
    if any(k in products_str for k in L3_KW): return 3
    elif any(k in products_str for k in L2_KW): return 2
    return 1

df['arch_level_est'] = df['Products Used'].fillna('').apply(estimate_level)

print(f"Loaded {len(df)} stories. Setup complete — run any block below.")

Mounted at /content/drive
Analytical sample after removing unavailable stories: 900 stories
Loaded 900 stories. Setup complete — run any block below.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Derive Analytical Columns

In [3]:
# ============================================================
# BLOCK 2.2 — Products by Triadic Architecture Layer
# ============================================================
# Block 2 shows the top 15 individual products by mention count.
# This block categorises ALL 282 unique product names into the
# three layers of the Triadic Architecture (Van der Vlist et al.,
# 2024) and shows the distribution of mentions across layers.
#
# Layer 1 — Foundational (Models & AI Services)
#   Raw cognitive power: AI models, ML platforms, cognitive APIs.
#   Products the customer cannot see but which enable everything.
#   Examples: Azure OpenAI Service, Azure AI Foundry, Azure ML
#
# Layer 2 — Infrastructural (Cloud Platform, Data & DevOps)
#   Hosting, compute, storage, data pipelines, security backbone.
#   The environment in which AI and business logic runs.
#   Examples: Azure, Azure Cosmos DB, Azure Databricks, GitHub
#
# Layer 3 — Deployment (Copilots, Apps, Productivity & Low-code)
#   User-facing interfaces where AI becomes "appified" — the
#   surface the enterprise user actually touches.
#   Examples: M365 Copilot, Copilot Studio, Power Apps, Teams
#
# Theoretical link: each layer maps directly onto the Triadic
# Architecture. A customer using products from all three layers
# is fully entangled — the core of the "Great House" lock-in thesis.
# ============================================================

from collections import Counter

LAYER_MAP = {
    # ── Layer 1: Foundational (Models & AI Services) ──────────────────────
    'Azure OpenAI Service':1,'Azure OpenAI':1,'Azure AI Foundry':1,
    'Azure AI Services':1,'Azure AI Studio':1,'Azure AI and Machine Learning':1,
    'Azure Machine Learning':1,'Azure AI Search':1,'Azure AI Document Intelligence':1,
    'Azure AI Speech':1,'Azure AI Language':1,'Azure AI Vision':1,
    'Azure AI Bot Service':1,'Azure AI Content Safety':1,'Azure AI Translator':1,
    'Azure AI Foundry Models':1,'Azure AI Model Catalog':1,'Azure AI Video Indexer':1,
    'Azure AI Content Understanding':1,'Azure AI Agent Service':1,
    'Azure AI Foundry Agent Service':1,'Azure AI Service':1,
    'Azure AI Immersive Reader':1,'Azure AI Metrics Advisor':1,
    'Azure AI and Machine Learning (AI)':1,
    'Azure OpenAI in Foundry Models':1,'Azure Document Intelligence in Foundry Tools':1,
    'Azure Speech in Foundry Tools':1,'Foundry Models':1,'Foundry Tools':1,
    'Microsoft Foundry':1,'Agents':1,'Azure Phi':1,
    'Bing Search APIs':1,'Bing':1,'Azure Face':1,'Azure Open AI Service':1,
    'Azure Machine Learning studio':1,'Azure Open Datasets':1,
    'Microsoft Dragon Copilot':1,

    # ── Layer 2: Infrastructural (Cloud Platform, Data, DevOps, Security) ─
    'Azure':2,'Azure App Service':2,'Azure Kubernetes Service':2,
    'Azure Container Apps':2,'Azure Container Registry':2,
    'Azure Container Instances':2,'Azure Functions':2,
    'Azure Virtual Machines':2,'Azure Virtual Desktop':2,
    'Azure Virtual Machine Scale Sets':2,'Azure VMware Solution':2,
    'Azure Blob Storage':2,'Azure SQL Database':2,'Azure SQL':2,
    'Azure SQL Managed Instance':2,'Azure SQL Edge':2,
    'SQL Server on Azure Virtual Machines':2,
    'Azure Cosmos DB':2,'Azure Databricks':2,'Azure Data Factory':2,
    'Azure Data Lake Storage':2,'Azure Data Lake Analytics':2,
    'Azure Data Explorer':2,'Azure Data Box':2,'Azure Synapse Analytics':2,
    'Azure HDInsight':2,'Azure HDInsight on Azure Kubernetes Service (AKS)':2,
    'Azure Database for PostgreSQL':2,'Azure Database for MySQL':2,
    'Azure Cache for Redis':2,'Azure Managed Redis':2,'Azure Files':2,
    'Azure Storage':2,'Azure Ultra Disk Storage':2,'Azure Table Storage':2,
    'Azure Backup':2,'Azure Site Recovery':2,'Azure Arc':2,
    'Azure Stack':2,'Azure Stack HCI':2,'Azure Local':2,
    'Azure Monitor':2,'Azure Application Insights':2,'Azure Resource Manager':2,
    'Cost Management':2,'Azure Service Bus':2,'Azure Event Hubs':2,
    'Azure Service Fabric':2,'Azure Logic Apps':2,'Azure API Management':2,
    'Azure Front Door':2,'Azure Application Gateway':2,
    'Azure Web Application Firewall':2,'Azure Traffic Manager':2,
    'Azure Load Balancer':2,'Azure ExpressRoute':2,'Azure VPN Gateway':2,
    'Azure Private Link':2,'Azure Bastion':2,'Azure DDoS Protection':2,
    'Azure Firewall':2,'Azure Virtual Network':2,
    'Azure Content Delivery Network':2,'Azure Static Web Apps':2,
    'Azure Web':2,'Azure Cloud Services':2,'Azure Compute':2,
    'Azure Compute Gallery':2,'Azure HPC':2,'Azure Communication Services':2,
    'Azure SignalR Service':2,'Azure Notification Hubs':2,
    'Azure Stream Analytics':2,'Azure Time Series Insights':2,'Azure Maps':2,
    'Azure IoT Hub':2,'Azure IoT Edge':2,'Azure Internet of Things':2,
    'Azure NetApp Files':2,'Azure Information Protection':2,
    'Azure Health Data Services':2,'Azure Red Hat OpenShift':2,
    'Azure Deployment Environments':2,'Azure Pipelines':2,'Azure DevOps':2,
    'Azure SDK':2,'Azure Analytics':2,'Azure Databases':2,'Azure Security':2,
    'Azure Kubernetes Services on Vmware':2,
    'Azure Key Vault':2,'Azure Managed Identity Services':2,
    'Azure Operator Call Protection':2,
    'Linux on Azure':2,'SAP on Azure':2,'Oracle on Azure':2,
    'Oracle Database at Azure':2,'Windows Server':2,
    'GitHub':2,'GitHub Advanced Security for Azure DevOps':2,
    'Visual Studio':2,'Visual Studio Code':2,'.NET':2,'SQL Server':2,
    'Microsoft Entra ID':2,'Microsoft Entra':2,'Microsoft Entra External ID':2,
    'Microsoft Entra Verified ID':2,'Microsoft Entra ID Governance':2,
    'Microsoft Entra Permissions Management':2,
    'Microsoft Sentinel':2,'Microsoft Defender':2,
    'Microsoft Defender for Cloud':2,'Microsoft Defender for Cloud Apps':2,
    'Microsoft Defender for Endpoint':2,'Microsoft Defender for Identity':2,
    'Microsoft Defender for Business':2,'Microsoft Defender XDR':2,
    'Microsoft Defender Threat Intelligence':2,
    'Microsoft Purview':2,'Microsoft Purview Information Protection':2,
    'Microsoft Purview Data Loss Prevention':2,'Microsoft Purview eDiscovery':2,
    'Microsoft Purview Data Governance App':2,
    'Intelligent Data Platform':2,'OneLake':2,'Microsoft Fabric':2,
    'Azure Analysis Services':2,'Real-Time Analytics':2,
    'Microsoft Azure Data Manager for Agriculture':2,

    # ── Layer 3: Deployment (Copilots, Apps, Productivity, Low-code) ──────
    'Microsoft 365 Copilot':3,'Microsoft Copilot':3,
    'Microsoft Copilot for Microsoft 365':3,'Microsoft Copilot Studio':3,
    'Microsoft Copilot for Sales':3,'Microsoft Copilot for Service':3,
    'Microsoft Copilot for Security':3,'Microsoft Security Copilot':3,
    'Microsoft Copilot for Dynamics 365':3,'Microsoft Copilot for Azure':3,
    'Microsoft Copilot in Azure':3,'Microsoft 365 Copilot for Sales':3,
    'Microsoft 365 Copilot for Service':3,'Microsoft 365 Copilot for Finance':3,
    'Microsoft 365 Copilot Chat':3,'GitHub Copilot':3,
    'Microsoft Teams':3,'Microsoft 365':3,'Microsoft 365 Enterprise':3,
    'Microsoft 365 E3':3,'Microsoft 365 E5':3,'Microsoft 365 A3':3,
    'Microsoft 365 A5':3,'Microsoft 365 F1':3,'Microsoft 365 Business':3,
    'Microsoft 365 Business Premium':3,'Microsoft 365 Government G5':3,
    'Microsoft 365 Frontline Worker':3,
    'Power BI':3,'Power BI Embedded':3,'Power Apps':3,
    'Power Automate':3,'Power Pages':3,'Microsoft Power Platform':3,
    'SharePoint':3,'SharePoint Online':3,'OneDrive':3,'OneDrive for Business':3,
    'Outlook':3,'Excel':3,'Word':3,'PowerPoint':3,'Forms':3,
    'OneNote':3,'Whiteboard':3,'Office':3,'Microsoft Stream':3,
    'Microsoft Teams Phone':3,'Microsoft Teams Rooms':3,
    'Dataverse':3,'Dataverse for Teams':3,
    'Dynamics 365':3,'Dynamics 365 Customer Service':3,'Dynamics 365 Sales':3,
    'Dynamics 365 Finance':3,'Dynamics 365 Contact Center':3,
    'Dynamics 365 Customer Insights':3,'Dynamics 365 Customer Insights - Data':3,
    'Dynamics 365 Supply Chain Management':3,'Dynamics 365 Field Service':3,
    'Dynamics 365 Commerce':3,'Dynamics 365 Business Central':3,
    'Dynamics 365 Human Resources':3,'Dynamics 365 Project Operations':3,
    'Dynamics 365 AI':3,'Dynamics 365 Customer Voice':3,
    'Dynamics 365 Fraud Protection':3,'Dynamics CRM':3,
    'Microsoft Intune':3,'Windows Autopilot':3,'Windows 11':3,
    'Windows 365':3,'Windows 365 Link':3,'Windows Hello':3,'Windows':3,
    'Surface':3,'Surface Laptop':3,'Surface Pro':3,'Surface Go':3,
    'Microsoft Viva':3,'Microsoft Viva Insights':3,'Microsoft Viva Engage':3,
    'Microsoft Viva Learning':3,'Microsoft Viva Suite':3,
    'Microsoft Viva Connections':3,'Microsoft Viva Pulse':3,
    'Microsoft Viva Goals':3,'Microsoft Viva Glint':3,
    'Microsoft Graph':3,'Microsoft Bot Framework':3,'Microsoft Learn':3,
    'Microsoft Reflect':3,'Microsoft Sustainability Manager':3,
    'Microsoft Marketplace':3,'Microsoft commercial marketplace':3,
    'Azure Marketplace':3,'LinkedIn':3,'LinkedIn Sales Navigator':3,
    'Microsoft Relationship Sales':3,'Planner':3,
    'Exchange Online':3,'Exchange':3,
    'Microsoft Endpoint Configuration Manager':3,'Microsoft Dev Box':3,
    'Minecraft':3,'Clarity':3,'Eventhouse':3,'Microsoft Translator':3,
    'Microsoft Edge':3,'Microsoft':3,'Apple':3,'Google':3,
}

# ── Compute layer totals ──────────────────────────────────────────────────────
all_products_flat = []
for _, row in df.iterrows():
    prods = [p.strip() for p in str(row['Products Used']).split('\n') if p.strip()]
    all_products_flat.extend(prods)

prod_counts = Counter(all_products_flat)
total_mentions = sum(prod_counts.values())

layer_mentions  = {1: 0, 2: 0, 3: 0}
layer_products  = {1: [], 2: [], 3: []}
layer_uniq      = {1: 0, 2: 0, 3: 0}

for prod, count in prod_counts.items():
    layer = LAYER_MAP.get(prod, None)
    if layer:
        layer_mentions[layer]  += count
        layer_products[layer].append((prod, count))
        layer_uniq[layer] += 1

# Stories touching each layer (at least 1 product from that layer)
layer_story_counts = {1:0, 2:0, 3:0}
for _, row in df.iterrows():
    prods = {p.strip() for p in str(row['Products Used']).split('\n') if p.strip()}
    for l in [1,2,3]:
        if any(LAYER_MAP.get(p) == l for p in prods):
            layer_story_counts[l] += 1

print(f'Total mentions: {total_mentions}')
for l in [1,2,3]:
    print(f'  Layer {l}: {layer_mentions[l]} mentions | '
          f'{layer_story_counts[l]} stories ({layer_story_counts[l]/len(df)*100:.0f}%) | '
          f'{layer_uniq[l]} unique products')

# ── Cross-layer entanglement per story ────────────────────────────────────────
def count_layers(products_str):
    prods = {p.strip() for p in str(products_str).split('\n') if p.strip()}
    layers_present = set()
    for p in prods:
        l = LAYER_MAP.get(p)
        if l:
            layers_present.add(l)
    return len(layers_present)

df['layers_touched'] = df['Products Used'].apply(count_layers)
layer_touch_dist = df['layers_touched'].value_counts().sort_index()

print('\nCross-layer entanglement per story:')
for k, v in layer_touch_dist.items():
    label = {0:'No mapped products', 1:'Single layer', 2:'Two layers', 3:'All 3 layers'}.get(k, f'{k} layers')
    print(f'  {label}: {v} stories ({v/len(df)*100:.1f}%)')


Total mentions: 4104
  Layer 1: 947 mentions | 534 stories (59%) | 40 unique products
  Layer 2: 1480 mentions | 566 stories (63%) | 130 unique products
  Layer 3: 1677 mentions | 571 stories (63%) | 112 unique products

Cross-layer entanglement per story:
  Single layer: 296 stories (32.9%)
  Two layers: 437 stories (48.6%)
  All 3 layers: 167 stories (18.6%)


In [4]:
# ============================================================
# BLOCK 3 — Estimated AI Embeddedness Level (arch_level_est)
# ============================================================
# IMPORTANT: arch_level_est is a HEURISTIC ESTIMATE only.
# It is NOT the final arch_level variable.
# Final arch_level requires LLM coding using both product signal
# AND usage signal from story content (Section 3.5 of methods).
#
# What changed from the original Block 3:
#   Removed from L3 keywords: Azure Container, Azure Compute,
#   Azure Blob, Azure Cosmos, Azure Data, Azure Function,
#   Azure Logic, Azure Kubernetes, Azure ML.
#   These are NEUTRAL infrastructure products — they appear at
#   all three levels and do not signal embeddedness depth.
#   Only genuine custom AI development products trigger L3.
# ============================================================

# ── Level 3 keywords — custom AI development products only ────────────────────
L3_KW = [
    # Azure AI Foundry family
    'Azure AI Foundry', 'Azure AI Foundry Models', 'Azure AI Foundry Agent Service',
    'Azure OpenAI in Foundry Models', 'Azure Document Intelligence in Foundry Tools',
    'Azure Speech in Foundry Tools', 'Microsoft Foundry', 'Foundry Models', 'Foundry Tools',
    # Azure OpenAI — direct API / custom model access
    'Azure OpenAI Service', 'Azure OpenAI', 'Azure Open AI Service',
    # Azure Machine Learning — custom model training and deployment
    'Azure Machine Learning', 'Azure Machine Learning studio',
    # Azure AI cognitive services — all specific AI capability APIs
    'Azure AI Services', 'Azure AI Studio', 'Azure AI and Machine Learning',
    'Azure AI Search', 'Azure AI Document Intelligence', 'Azure AI Speech',
    'Azure AI Language', 'Azure AI Vision', 'Azure AI Bot Service',
    'Azure AI Content Safety', 'Azure AI Translator', 'Azure AI Video Indexer',
    'Azure AI Content Understanding', 'Azure AI Agent Service', 'Azure AI Service',
    'Azure AI Immersive Reader', 'Azure AI Metrics Advisor', 'Azure AI Model Catalog',
    'Azure AI and Machine Learning (AI)',
    # Other foundational AI products
    'Agents', 'Azure Phi', 'Microsoft Dragon Copilot',
    'Azure Open Datasets', 'Microsoft Azure Data Manager for Agriculture',
]

# ── Level 2 keywords — low-code AI orchestration ─────────────────────────────
# Checked only if NO Level 3 signal is present in the story.
L2_KW = [
    'Copilot Studio',    # AI agent builder with business logic integration
    'Power Automate',    # automated workflow orchestration
    'Power Apps',        # low-code application builder
    'Power Platform',    # catches Microsoft Power Platform as a whole
    'Power Pages',       # low-code web portal builder
    'Dataverse',         # structured data layer for Power Platform
]

# ── Level 1 — DEFAULT ─────────────────────────────────────────────────────────
# Assigned when NO Level 3 or Level 2 signal is present.
# Includes: all Copilot-for-[Product] variants (M365 Copilot, GitHub Copilot,
# Copilot for Sales/Service/Security/Dynamics 365 etc.), Microsoft Copilot,
# and all neutral infrastructure / productivity products (Azure, Teams, Power BI,
# SharePoint, Dynamics 365, Microsoft 365, GitHub, Microsoft Entra etc.)
# No keyword needed — Level 1 is the explicit default fallback.

def estimate_level(products_str):
    if any(k in products_str for k in L3_KW): return 3
    elif any(k in products_str for k in L2_KW): return 2
    return 1  # Level 1 default

df['arch_level_est'] = df['Products Used'].fillna('').apply(estimate_level)
level_counts = df['arch_level_est'].value_counts().sort_index()

print('Estimated arch level distribution (corrected keywords):')
for l in [1, 2, 3]:
    n = level_counts.get(l, 0)
    print(f'  Level {l}: {n} ({n/len(df)*100:.1f}%)')


Estimated arch level distribution (corrected keywords):
  Level 1: 267 (29.7%)
  Level 2: 100 (11.1%)
  Level 3: 533 (59.2%)


In [5]:
# ============================================================
# BLOCK 4.2 — Industry Distribution (reclassified, n=900)
# ============================================================
# This block builds on Block 4 by manually reclassifying the
# 100 stories that carried Microsoft catch-all industry labels
# ("Commercial Other Industries", "Other - Unsegmented", etc.)
# or had no label at all.
#
# Methodology:
#   - Reclassification is based on company name and domain knowledge
#   - Every decision is explicit in the lookup table below
#   - The result is compared side-by-side with Block 4 to show
#     what the reclassification changes
#   - 4 stories remain unresolvable → kept as Other / Unclassified
#   - This block is additive — Block 4 is NOT replaced
# ============================================================

RECLASSIFY_MAP = {
    # ── Financial Services ───────────────────────────────────────────────────
    'ABN AMRO Bank':                            'Financial Services',
    'Kuwait Finance House':                     'Financial Services',
    'Banco de la Ciudad de Buenos Aires':       'Financial Services',
    'Intesa Sanpaolo Group':                    'Financial Services',
    'Virgin Money':                             'Financial Services',
    'DIBANKA':                                  'Financial Services',
    'Commercial Bank of Dubai':                 'Financial Services',
    'ClearBank':                                'Financial Services',
    'National Australia Bank':                  'Financial Services',
    'Nest Bank':                                'Financial Services',
    'Paysafe':                                  'Financial Services',
    'Cradle Fund':                              'Financial Services',
    'CSOP ASSET MANAGEMENT LTD':               'Financial Services',
    'Permodalan Nasional Berhad':               'Financial Services',
    'Absa':                                     'Financial Services',

    # ── Professional Services ────────────────────────────────────────────────
    'WinWire Technologies':                     'Professional Services',
    'Persistent Systems':                       'Professional Services',
    'Talan':                                    'Professional Services',
    'baseVISION AG':                            'Professional Services',
    'Eide Bailly':                              'Professional Services',
    'Cognizant Technology Solutions India Pvt Ltd': 'Professional Services',
    'GroupeActive':                             'Professional Services',
    'AnalogFolk Marmite Tasteface':             'Professional Services',
    'ALLEGIS GROUP':                            'Professional Services',
    'legal i':                                  'Professional Services',
    'OneDigital':                               'Professional Services',
    'Amey':                                     'Professional Services',
    'MCI Group':                                'Professional Services',
    'Adecco Group AG':                          'Professional Services',
    'Mitsui':                                   'Professional Services',
    'Lionbridge Technologies':                  'Professional Services',
    'EY Global':                                'Professional Services',
    'MHP':                                      'Professional Services',

    # ── Technology ───────────────────────────────────────────────────────────
    'Relativity':                               'Technology',
    'OneTrust':                                 'Technology',
    'Icertis':                                  'Technology',
    'ServiceNow':                               'Technology',
    'Cognition AI':                             'Technology',
    'Weights & Biases':                         'Technology',
    'Weights and Biases':                       'Technology',
    'LlamaIndex':                               'Technology',
    'Faros AI':                                 'Technology',
    'Cassidy':                                  'Technology',
    'Agora':                                    'Technology',
    'Space and Time':                           'Technology',
    'Agnostic Intelligence':                    'Technology',
    'EcoVadis':                                 'Technology',
    'Zenya':                                    'Technology',
    'WeTransact':                               'Technology',
    'Giatec Scientific':                        'Technology',
    'Hello Lamp Post':                          'Technology',
    'ASC Technologies AG':                      'Technology',
    'AVEVA':                                    'Technology',
    'Profisee Group INC':                       'Technology',
    'Kinectify':                                'Technology',
    'GigXR':                                    'Technology',
    'Transact Campus':                          'Technology',
    'Fujitsu':                                  'Technology',
    'HYPE':                                     'Technology',
    'elunic':                                   'Technology',
    'Sensei':                                   'Technology',
    'ESW':                                      'Technology',
    'Microsoft':                                'Technology',

    # ── Manufacturing ────────────────────────────────────────────────────────
    'PARKER HANNIFIN CORP':                     'Manufacturing',
    'NXP Semiconductors':                       'Manufacturing',
    'Huber+Suhner AG':                          'Manufacturing',
    'Victorinox AG':                            'Manufacturing',
    'Syensqo':                                  'Manufacturing',
    'Topsoe':                                   'Manufacturing',
    'Mitsubishi Heavy Industries, Ltd.':        'Manufacturing',
    'Wilo SE':                                  'Manufacturing',
    'ALSTOM TRANSPORT':                         'Manufacturing',
    'LEXMARK INTERNATIONAL INC':               'Manufacturing',
    'Eastman':                                  'Manufacturing',

    # ── Retail & Consumer ────────────────────────────────────────────────────
    "L'Oréal":                                  'Retail & Consumer',
    "L'Oreal":                                  'Retail & Consumer',
    "‘L’Oréal’":                               'Retail & Consumer',
    "’L’Oréal’":                               'Retail & Consumer',
    'L’Oréal':                                   'Retail & Consumer',
    'StarKist Foods':                           'Retail & Consumer',
    'Reckitt':                                  'Retail & Consumer',
    'Carlsberg Group':                          'Retail & Consumer',
    'Lotte Hotels and Resorts':                 'Retail & Consumer',
    'Cdiscount':                                'Retail & Consumer',
    'Chin Hin Group Berhad':                    'Retail & Consumer',
    'Alpitour World':                           'Retail & Consumer',

    # ── Transport & Logistics ────────────────────────────────────────────────
    'CH Robinson':                              'Transport & Logistics',
    'Evri':                                     'Transport & Logistics',
    'Wallenius Wilhelmsen':                     'Transport & Logistics',
    'World2Meet':                               'Transport & Logistics',
    'Miral':                                    'Transport & Logistics',
    'Voltas Limited':                           'Transport & Logistics',

    # ── Real Estate & Construction ───────────────────────────────────────────
    'Sinyi Realty':                             'Real Estate & Construction',
    'KVL Bauconsult':                           'Real Estate & Construction',
    'Balfour Beatty':                           'Real Estate & Construction',
    'Arada':                                    'Real Estate & Construction',
    'HOUSE730':                                 'Real Estate & Construction',
    'CapitaLand Investment':                    'Real Estate & Construction',

    # ── Energy & Utilities ───────────────────────────────────────────────────
    'BKW FMB Energie AG':                       'Energy & Utilities',
    'Theodora':                                 'Energy & Utilities',

    # ── Education ────────────────────────────────────────────────────────────
    'Saudia Academy':                           'Education',
}

# ── Apply reclassification on top of existing industry_grouped ───────────────
OTHER_LABELS = {'Commercial Other Industries', 'Unknown',
                'Other - Unsegmented', 'Other Services'}

def reclassify(row):
    """
    If the story was in an Other/Unclassified group, check whether
    the company name appears in RECLASSIFY_MAP. If yes, use the mapped
    sector. If no match, keep as Other / Unclassified.
    """
    if row['industry_grouped'] == 'Other / Unclassified':
        name = row['Customer Name'].strip()
        return RECLASSIFY_MAP.get(name, 'Other / Unclassified')
    return row['industry_grouped']

df['industry_reclassified'] = df.apply(reclassify, axis=1)

# ── Stats ─────────────────────────────────────────────────────────────────────
before = (df['industry_grouped'] == 'Other / Unclassified').sum()
after  = (df['industry_reclassified'] == 'Other / Unclassified').sum()
print(f"Other / Unclassified: {before} → {after} after reclassification")
print(f"Successfully reclassified: {before - after} stories\n")

ind_counts_v2 = df['industry_reclassified'].value_counts()

# ── Print final distribution ──────────────────────────────────────────────────
print('Final industry distribution after reclassification:')
print(f'{"Sector":<35} {"n":>5}  {"% of corpus":>12}')
print('-' * 56)
for sector, cnt in ind_counts_v2.items():
    marker = '  ← flagged' if sector == 'Other / Unclassified' else ''
    print(f'{sector:<35} {cnt:>5}  {cnt/len(df)*100:>11.1f}%{marker}')


Other / Unclassified: 100 → 0 after reclassification
Successfully reclassified: 100 stories

Final industry distribution after reclassification:
Sector                                  n   % of corpus
--------------------------------------------------------
Professional Services                 176         19.6%
Financial Services                    117         13.0%
Manufacturing                          94         10.4%
Retail & Consumer                      90         10.0%
Health                                 79          8.8%
Telecom & Media                        70          7.8%
Technology                             64          7.1%
Education                              45          5.0%
Nonprofit & IGO                        42          4.7%
Government & Public Sector             39          4.3%
Energy & Utilities                     34          3.8%
Transport & Logistics                  28          3.1%
Real Estate & Construction             22          2.4%


In [6]:
# Recompute triadic_layers_count and arch_level_est from the product stack.
# (Column derivation only — the Block 8 exploratory comparison has been removed.)

# ============================================================
# BLOCK 8 — Triadic Architecture Count vs Heuristic Arch Level
# ============================================================
# Compares two variables derived from the SAME source:
#   triadic_layers_count — how many architectural layers
#                          (Foundational/Infrastructural/Deployment)
#                          are represented in the product stack
#   arch_level_est       — heuristic embeddedness level (1/2/3)
#                          based on which specific AI products appear
#
# Both use only the Products Used field — no LLM needed.
# Purpose: show the structural relationship between product-layer
# breadth and embeddedness depth BEFORE story content is read.
# Limitation: because both come from the same source, they will
# correlate strongly. The truly interesting comparison is
# triadic_layers_count × LLM-coded arch_level (done after coding).
#
# Key finding preview: 310 stories (34.4%) have 2 triadic layers
# but heuristic L3 — showing the heuristic classifies many
# "partial stack" companies as deeply embedded.
# ============================================================

from collections import Counter
import numpy as np

# ── Recompute both variables ──────────────────────────────────────────────────
LAYER_MAP = {
    # Layer 1 — Foundational (AI models & cognitive services)
    'Azure OpenAI Service':1,'Azure OpenAI':1,'Azure AI Foundry':1,
    'Azure AI Services':1,'Azure AI Studio':1,'Azure AI and Machine Learning':1,
    'Azure Machine Learning':1,'Azure Machine Learning studio':1,
    'Azure AI Search':1,'Azure AI Document Intelligence':1,'Azure AI Speech':1,
    'Azure AI Language':1,'Azure AI Vision':1,'Azure AI Bot Service':1,
    'Azure AI Content Safety':1,'Azure AI Translator':1,'Azure AI Foundry Models':1,
    'Azure AI Model Catalog':1,'Azure AI Video Indexer':1,'Azure AI Content Understanding':1,
    'Azure AI Agent Service':1,'Azure AI Foundry Agent Service':1,'Azure AI Service':1,
    'Azure AI Immersive Reader':1,'Azure AI Metrics Advisor':1,
    'Azure AI and Machine Learning (AI)':1,'Azure OpenAI in Foundry Models':1,
    'Azure Document Intelligence in Foundry Tools':1,'Azure Speech in Foundry Tools':1,
    'Foundry Models':1,'Foundry Tools':1,'Microsoft Foundry':1,'Agents':1,
    'Azure Phi':1,'Microsoft Dragon Copilot':1,'Azure Open AI Service':1,
    'Azure Machine Learning studio':1,'Azure Open Datasets':1,
    'Microsoft Azure Data Manager for Agriculture':1,
    # Layer 2 — Infrastructural (cloud, data, DevOps, security)
    'Azure':2,'Azure App Service':2,'Azure Kubernetes Service':2,'Azure Container Apps':2,
    'Azure Container Registry':2,'Azure Functions':2,'Azure Virtual Machines':2,
    'Azure Blob Storage':2,'Azure SQL Database':2,'Azure SQL':2,
    'Azure Cosmos DB':2,'Azure Databricks':2,'Azure Data Factory':2,
    'Azure Data Lake Storage':2,'Azure Synapse Analytics':2,'Azure Database for PostgreSQL':2,
    'Azure Database for MySQL':2,'Azure Cache for Redis':2,'Azure Files':2,
    'Azure Storage':2,'Azure Monitor':2,'Azure API Management':2,'Azure Logic Apps':2,
    'Azure Service Bus':2,'Azure Event Hubs':2,'Azure DevOps':2,'Azure Key Vault':2,
    'Azure Operator Call Protection':2,'Azure Managed Identity Services':2,
    'Azure IoT Hub':2,'Azure IoT Edge':2,'Azure Communication Services':2,
    'Azure Stream Analytics':2,'Azure Data Explorer':2,'Azure Backup':2,
    'Azure Firewall':2,'Azure VPN Gateway':2,'Azure Bastion':2,'Azure Arc':2,
    'Azure Stack':2,'Azure HPC':2,'Azure Compute':2,'Azure Compute Gallery':2,
    'Azure Virtual Network':2,'Azure Managed Redis':2,'Azure Container Instances':2,
    'GitHub':2,'Visual Studio':2,'Visual Studio Code':2,'SQL Server':2,
    'Microsoft Entra ID':2,'Microsoft Entra':2,'Microsoft Entra External ID':2,
    'Microsoft Entra Verified ID':2,'Microsoft Sentinel':2,'Microsoft Defender':2,
    'Microsoft Defender for Cloud':2,'Microsoft Defender for Endpoint':2,
    'Microsoft Defender XDR':2,'Microsoft Defender for Business':2,
    'Microsoft Defender Threat Intelligence':2,'Microsoft Defender for Cloud Apps':2,
    'Microsoft Defender for Identity':2,'Microsoft Purview':2,
    'Microsoft Purview Information Protection':2,'Intelligent Data Platform':2,
    'OneLake':2,'Microsoft Fabric':2,'Real-Time Analytics':2,
    'Linux on Azure':2,'SAP on Azure':2,'Oracle on Azure':2,'Windows Server':2,'.NET':2,
    # Layer 3 — Deployment (copilots, apps, low-code, productivity)
    'Microsoft 365 Copilot':3,'Microsoft Copilot':3,'Microsoft Copilot Studio':3,
    'Microsoft Copilot for Microsoft 365':3,'Microsoft Copilot for Sales':3,
    'Microsoft Copilot for Service':3,'Microsoft Copilot for Security':3,
    'Microsoft Security Copilot':3,'Microsoft Copilot for Dynamics 365':3,
    'Microsoft 365 Copilot Chat':3,'GitHub Copilot':3,'Microsoft Teams':3,
    'Microsoft 365':3,'Power BI':3,'Power Apps':3,'Power Automate':3,
    'Microsoft Power Platform':3,'SharePoint':3,'OneDrive':3,'Outlook':3,
    'Excel':3,'Word':3,'PowerPoint':3,'Dataverse':3,'Power Pages':3,
    'Dynamics 365':3,'Dynamics 365 Customer Service':3,'Dynamics 365 Sales':3,
    'Dynamics 365 Finance':3,'Dynamics 365 Contact Center':3,
    'Dynamics 365 Customer Insights':3,'Dynamics 365 Supply Chain Management':3,
    'Dynamics 365 Field Service':3,'Dynamics 365 Commerce':3,
    'Dynamics 365 Business Central':3,'Dynamics 365 Human Resources':3,
    'Dynamics 365 Project Operations':3,'Dynamics 365 AI':3,'Dynamics CRM':3,
    'Microsoft Intune':3,'Windows 11':3,'Windows 365':3,
    'Microsoft Viva':3,'Microsoft Viva Insights':3,'Microsoft Viva Engage':3,
    'Microsoft Graph':3,'Microsoft Edge':3,'Microsoft':3,
    'Dynamics 365 Customer Insights - Data':3,'Dynamics 365 Customer Voice':3,
}

L3_KW = [
    'Azure AI Foundry','Azure AI Foundry Models','Azure AI Foundry Agent Service',
    'Azure OpenAI in Foundry Models','Azure Document Intelligence in Foundry Tools',
    'Azure Speech in Foundry Tools','Microsoft Foundry','Foundry Models','Foundry Tools',
    'Azure OpenAI Service','Azure OpenAI','Azure Open AI Service',
    'Azure Machine Learning','Azure Machine Learning studio',
    'Azure AI Services','Azure AI Studio','Azure AI and Machine Learning',
    'Azure AI Search','Azure AI Document Intelligence','Azure AI Speech',
    'Azure AI Language','Azure AI Vision','Azure AI Bot Service',
    'Azure AI Content Safety','Azure AI Translator','Azure AI Video Indexer',
    'Azure AI Content Understanding','Azure AI Agent Service','Azure AI Service',
    'Azure AI Immersive Reader','Azure AI Metrics Advisor','Azure AI Model Catalog',
    'Azure AI and Machine Learning (AI)','Agents','Azure Phi',
    'Microsoft Dragon Copilot','Azure Open Datasets',
    'Microsoft Azure Data Manager for Agriculture',
]
L2_KW = ['Copilot Studio','Power Automate','Power Apps','Power Platform',
         'Power Pages','Dataverse']

def get_triadic(products_str):
    prods = [p.strip() for p in products_str.split('\n') if p.strip()]
    layers = set(LAYER_MAP.get(p) for p in prods if LAYER_MAP.get(p))
    return len(layers)

def get_heuristic(products_str):
    if any(k in products_str for k in L3_KW): return 3
    elif any(k in products_str for k in L2_KW): return 2
    return 1

df['triadic_layers_count'] = df['Products Used'].fillna('').apply(get_triadic)
df['arch_level_est'] = df['Products Used'].fillna('').apply(get_heuristic)

print("Derived: triadic_layers_count, arch_level_est")
print(df[['triadic_layers_count','arch_level_est']].apply(lambda c: c.value_counts()).to_string())


Derived: triadic_layers_count, arch_level_est
   triadic_layers_count  arch_level_est
0                     2             NaN
1                   304           267.0
2                   440           100.0
3                   154           533.0


## Cleaned Datasets and Export

In [7]:
# ============================================================
# Export Clean Datasets
# Run AFTER Block 8 — df must have all computed columns
# ============================================================
# Requires these columns already in df (built in earlier blocks):
#   industry_reclassified  → Block 4.2
#   product_stack_breadth  → Block 2 / Block 0
#   triadic_layers_count   → Block 2.2
#   arch_level_est         → Block 3
#
# Produces two files saved to Google Drive:
#
# FILE 1: cleaned_scraped_900.csv
#   All pre-processed analytical columns — the "already done"
#   side of the master dataset. One row per story, 13 columns.
#   Use this as the base for merging LLM-coded results after
#   Block 9 (the LLM pipeline).
#
# FILE 2: pipeline_input_900.csv
#   Lean 4-column file for the Block 9 LLM coding pipeline.
#   Only what the LLM needs — saves tokens, keeps API calls clean.
# ============================================================

OUTPUT_DIR = '/content/drive/MyDrive/Microsoft900cases/'

# ── Verify required columns exist ────────────────────────────────────────────
required = ['industry_reclassified', 'product_stack_breadth',
            'triadic_layers_count', 'arch_level_est']
missing = [c for c in required if c not in df.columns]
if missing:
    print(f'⚠ Missing columns: {missing}')
    print('Make sure Blocks 2.2, 3, and 4.2 have been run before this block.')
else:
    print(f'✓ All required columns present. df has {len(df)} rows.')

# ── FILE 1: cleaned_scraped_900.csv ───────────────────────────────────────────
# Selects and renames columns to clean analytical names.
# Computes primary_need (first-listed business need) if not already present.

# primary_need = first-listed value in Business Need field
df['primary_need'] = df['Business Need'].fillna('').apply(
    lambda x: x.strip().split('\n')[0].strip() if x.strip() else ''
)

scraped = df[[
    'Customer Name',
    'Story URL',
    'Industry',                  # raw Microsoft label — kept for reference
    'industry_reclassified',     # two-stage consolidated sector
    'Organization Size',
    'Country',
    'Business Need',             # full multi-value string — all needs
    'primary_need',              # first-listed need only
    'Products Used',             # full product list string
    'product_stack_breadth',     # count of distinct products
    'triadic_layers_count',      # count of triadic layers in product stack (0-3)
    'arch_level_est',            # heuristic arch level (1/2/3) — estimated only
]].copy()

# Rename to clean analytical column names
scraped.columns = [
    'company_name',
    'story_url',
    'industry_raw',
    'industry_reclassified',
    'org_size',
    'country',
    'business_need_all',
    'primary_need',
    'products_used',
    'product_stack_breadth',
    'triadic_layers_count',
    'arch_level_est',
]

# Save
scraped_path = OUTPUT_DIR + 'cleaned_scraped_900.csv'
scraped.to_csv(scraped_path, index=False, encoding='utf-8-sig')
print(f'\n✓ FILE 1 saved: cleaned_scraped_900.csv')
print(f'  Rows: {len(scraped)} | Columns: {len(scraped.columns)}')
print(f'  Columns: {list(scraped.columns)}')

# Quick sense check
print(f'\n  industry_reclassified — unique values: {scraped["industry_reclassified"].nunique()}')
print(f'  arch_level_est distribution:')
for lvl, cnt in scraped['arch_level_est'].value_counts().sort_index().items():
    print(f'    L{lvl}: {cnt} ({cnt/len(scraped)*100:.1f}%)')
print(f'  Stories with no country: {scraped["country"].isna().sum() + (scraped["country"]=="").sum()}')
print(f'  Stories with no primary_need: {(scraped["primary_need"]=="").sum()}')

# ── FILE 2: pipeline_input_900.csv ────────────────────────────────────────────
# Exactly 4 columns — what the LLM needs and nothing else.
# Keeps original column names to match the Block 9 pipeline code.

pipeline = df[[
    'Customer Name',
    'Products Used',
    'Story Content',
    'Executive summary',
]].copy()

# Check story content availability
empty_content = (pipeline['Story Content'].fillna('').str.strip() == '').sum()
empty_exec    = (pipeline['Executive summary'].fillna('').str.strip() == '').sum()
print(f'\n  Stories with empty Story Content: {empty_content}')
print(f'  Stories with empty Executive summary: {empty_exec}')
if empty_content > 0:
    print(f'  ⚠ {empty_content} stories have no Story Content — LLM will code these as thin content')

# Save
pipeline_path = OUTPUT_DIR + 'pipeline_input_900.csv'
pipeline.to_csv(pipeline_path, index=False, encoding='utf-8-sig')
print(f'\n✓ FILE 2 saved: pipeline_input_900.csv')
print(f'  Rows: {len(pipeline)} | Columns: {list(pipeline.columns)}')

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"""
{'='*55}
BLOCK 8.5 COMPLETE
{'='*55}
Two files saved to: {OUTPUT_DIR}

cleaned_scraped_900.csv   ← base for master dataset
  • {len(scraped)} rows × {len(scraped.columns)} columns
  • All pre-processed scraped + computed variables
  • Ready to merge with LLM coding output after Block 9

pipeline_input_900.csv    ← input for Block 9 LLM pipeline
  • {len(pipeline)} rows × 4 columns
  • Customer Name, Products Used, Story Content, Executive summary
  • Use this as INPUT_FILE in Block 9

NEXT STEP:
  1. Test Block 9 prompt manually in Claude.ai with 3-5 stories
  2. Set ANTHROPIC_API_KEY and run run_test(n=5)
  3. Inspect test output, then run full pipeline
  4. After pipeline completes, run Block 9b merge script
     to combine cleaned_scraped_900.csv + LLM output
     into master_dataset_900.csv
{'='*55}
""")

✓ All required columns present. df has 900 rows.

✓ FILE 1 saved: cleaned_scraped_900.csv
  Rows: 900 | Columns: 12
  Columns: ['company_name', 'story_url', 'industry_raw', 'industry_reclassified', 'org_size', 'country', 'business_need_all', 'primary_need', 'products_used', 'product_stack_breadth', 'triadic_layers_count', 'arch_level_est']

  industry_reclassified — unique values: 13
  arch_level_est distribution:
    L1: 267 (29.7%)
    L2: 100 (11.1%)
    L3: 533 (59.2%)
  Stories with no country: 442
  Stories with no primary_need: 0

  Stories with empty Story Content: 1
  Stories with empty Executive summary: 23
  ⚠ 1 stories have no Story Content — LLM will code these as thin content

✓ FILE 2 saved: pipeline_input_900.csv
  Rows: 900 | Columns: ['Customer Name', 'Products Used', 'Story Content', 'Executive summary']

BLOCK 8.5 COMPLETE
Two files saved to: /content/drive/MyDrive/Microsoft900cases/

cleaned_scraped_900.csv   ← base for master dataset
  • 900 rows × 12 columns
  • 

## LLM Coding Pipeline (reference only — not executed)

The cells below define the coding pipeline used to produce the 18 LLM-coded variables (Claude Sonnet 4.6, `claude-sonnet-4-6`, temperature=0). They are kept for transparency. Execution calls are commented out — the coded output already exists in `master_dataset_900.csv`.

In [8]:
# Reference only — not executed.
# !pip install anthropic --break-system-packages

In [9]:
# API key setup (reference). Set your own key via environment variable when running.
# import os
# os.environ['ANTHROPIC_API_KEY'] = ''  # do NOT hardcode keys in shared notebooks

In [10]:
%%script false --no-raise-error
# (Reference only — this cell is not executed. The pipeline already ran; results are in master_dataset_900.csv.)
"""
=============================================================
BLOCK 9 — LLM Coding Pipeline  (FINAL — v3 prompt)
AI Appification Study · Microsoft Customer Stories (n=900)
=============================================================
Before running:
  1. !pip install anthropic --break-system-packages
  2. import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
  3. Run block8_5 first so pipeline_input_900.csv exists
  4. Always run run_test(n=5) before run_pipeline()
=============================================================
"""

import anthropic, json, csv, time, os
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL        = "claude-sonnet-4-6"
TEMPERATURE  = 0
MAX_TOKENS   = 2000
INPUT_FILE   = '/content/drive/MyDrive/Microsoft900cases/pipeline_input_900.csv'
OUTPUT_FILE  = '/content/drive/MyDrive/Microsoft900cases/llm_coded_results.jsonl'
ERROR_FILE   = '/content/drive/MyDrive/Microsoft900cases/llm_coding_errors.jsonl'
SLEEP        = 0.5   # seconds between calls — increase to 1.0 if rate limit hit

# ── SYSTEM PROMPT (v3 — final) ────────────────────────────────────────────────
SYSTEM_PROMPT = """
You are a precise research coder for an academic study on enterprise AI adoption.
Your job: read Microsoft customer stories and extract structured data exactly
as defined in the codebook below. Follow every rule exactly.

CORE RULES (apply to everything):
- Never invent or infer information not present in the story.
- Extract evidence sentences VERBATIM — word-for-word, no paraphrasing, no shortening.
- When in doubt between two values, always choose the LOWER / more conservative option.
- Return ONLY valid JSON. No explanation, no preamble, no markdown fences.

=================================================================
CODEBOOK — VARIABLE DEFINITIONS
=================================================================

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A1. arch_level  (integer: 1, 2, or 3)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Measures how deeply AI is integrated into the customer's operations.
CRITICAL: Do NOT assign based on product name alone.
You MUST verify the usage signal in the story content.

STEP 1 — Check for Level 3 product signal (check FIRST, overrides all):
Any of these products present → candidate for Level 3:
Azure AI Foundry, Azure AI Foundry Models, Azure AI Foundry Agent Service,
Azure OpenAI Service, Azure OpenAI, Azure Open AI Service,
Azure Machine Learning, Azure Machine Learning studio,
Azure AI Services, Azure AI Studio, Azure AI and Machine Learning,
Azure AI Search, Azure AI Document Intelligence, Azure AI Speech,
Azure AI Language, Azure AI Vision, Azure AI Bot Service,
Azure AI Content Safety, Azure AI Translator, Azure AI Video Indexer,
Azure AI Content Understanding, Azure AI Agent Service, Azure AI Service,
Azure AI Immersive Reader, Azure AI Metrics Advisor, Azure AI Model Catalog,
Azure AI and Machine Learning (AI), Azure OpenAI in Foundry Models,
Azure Document Intelligence in Foundry Tools, Azure Speech in Foundry Tools,
Microsoft Foundry, Foundry Models, Foundry Tools, Agents, Azure Phi,
Microsoft Dragon Copilot, Azure Open Datasets,
Microsoft Azure Data Manager for Agriculture

STEP 2 — Check for Level 2 product signal (only if no Level 3 present):
Any of these present → candidate for Level 2:
Microsoft Copilot Studio, Power Apps, Power Automate,
Microsoft Power Platform, Power Pages, Dataverse, Dataverse for Teams

STEP 3 — Default to Level 1 if no L3 or L2 signal present.
Neutral products (do NOT trigger any level):
M365 Copilot, GitHub Copilot, Microsoft Copilot, all Copilot-for-[Product]
variants, Azure, Teams, Power BI, SharePoint, Microsoft 365, Dynamics 365,
GitHub, Azure SQL, Azure Cosmos DB, Azure Blob Storage, Azure Functions,
Azure Container Apps, Azure Data Factory, Azure Logic Apps, Azure Kubernetes,
Azure Databricks, Azure App Service, Azure Virtual Machines, Microsoft Entra,
Microsoft Sentinel, Microsoft Purview, Microsoft Fabric, and all other
pure infrastructure or productivity products.

STEP 4 — Verify usage signal in story content:
LEVEL 3 confirmed if story describes: AI running autonomously, AI connected
to ERP/EHR/financial ledger/supply chain/core operational systems, customer
building OR deploying a Microsoft AI product deeply integrated into core
operational architecture.
NOTE: Pre-built Microsoft AI products (like Dragon Copilot) embedded
directly into EHR/ERP systems qualify as Level 3. The distinction is
between deep operational integration vs surface-level tool use.
LEVEL 2 confirmed if story describes: AI orchestrating multi-step workflows,
connecting to internal data (SharePoint, SQL, internal KB, CRM records),
human triggers overall process but AI executes steps within it.
LEVEL 1 confirmed if story describes: user manually invoking AI for single
discrete tasks (drafting, summarising, suggesting). No workflow orchestration.
No connection to core systems.

OVERRIDE RULE (always apply when reading story content):
Copilot Studio + ERP/EHR + autonomous operation described in story
→ assign Level 3 even if no Foundry/OpenAI product listed.

AMBIGUITY DEFAULTS:
- Between Level 1 and 2: assign Level 1 unless multi-step orchestration
  AND internal data connection are both clearly described.
- Between Level 2 and 3: assign Level 2 unless autonomous operation on
  CORE operational systems is clearly described.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A2. arch_level_signal_sentences  (list of 1–3 verbatim strings)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Extract 1–3 verbatim sentences from Story Content that PROVE the
arch_level assignment. Must show actual AI behaviour, system
integration, or operational usage.

DO extract: sentences describing autonomous AI operation, ERP/EHR
integration, custom AI construction, multi-step orchestration with
internal data, or explicit user-invoked single tasks.
DO NOT extract:
- Product name mentions alone ("The company uses Azure AI Foundry")
- Generic marketing claims ("AI transforms their business")
- Sentences from Executive Summary
- Sentences about non-AI tools unless that IS the primary AI deployment

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A3. agency_type  (string: "Tool" or "Agent")
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Tool: human explicitly invokes AI each time. Single discrete request.
AI produces output; human decides next action.
Signals: "I ask it to", "when I need it", "helps me write",
ambient recording that user activates.

Agent: AI acts proactively, executes multi-step tasks without
step-by-step human instruction, or makes autonomous decisions.
Signals: "automatically processes", "resolves tickets without human
intervention", "proactively flags", "orchestrates", "triggers when",
runs on schedule, multi-agent pipeline.

Counter-signals (do NOT assign Agent if only these):
"Copilot suggests the next step" (suggestion != autonomous action),
"we can ask it to do X" (still human-initiated),
automated batch processing pipelines where content arrives and the
system processes it automatically — this is automation, not agency.
Agent requires the AI to proactively DECIDE when and how to act,
not just run when triggered by data arrival or a scheduled job.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A4. data_connectivity  (string: "General", "Internal", or "System")
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Classify based on the PRIMARY AI deployment described in the story.

General: public knowledge only, no internal company data mentioned.

Internal: connects to company-specific INFORMATIONAL data —
  SharePoint, internal KB, email/Teams history, CRM contact records,
  pricing pages, intranet, internal wiki, Microsoft 365 data
  (meetings, emails, files, chats), security monitoring tools,
  SIEM platforms (Sentinel, Defender).

System: connects to TRANSACTIONAL/OPERATIONAL systems that actively
  run core business processes — ERP (SAP, Oracle), EHR (Epic, Cerner),
  financial ledger/accounting, supply chain management,
  HR/payroll (ADP), billing infrastructure, inventory management.

IMPORTANT:
- Cloud infrastructure provisioned at deploy time (Azure Container Apps,
  Azure Compute, databases) = NOT System. Platform resources are not
  enterprise operational systems.
- Security tools (Sentinel, Defender, SIEM, XDR) = Internal, not System.
- Microsoft 365, Teams, email, meetings, chats = Internal.
- CRM = Internal if contact records only; System only if it triggers
  automated operational transactions.
- If a story has multiple AI deployments, classify on the PRIMARY one.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
B1. roi_metric  (list — pick ALL that apply, exact strings only)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Use ONLY these exact strings — no variations:
"Efficiency"          — faster processes, time saved, automation
"Cost"                — money saved, cost reduction, consolidation savings
"Productivity"        — more output per person, higher throughput
"Quality"             — fewer errors, better accuracy, improved outcomes
"Employee Experience" — satisfaction, reduced burnout, meaningful work
"Customer Experience" — better service, faster response, improved outcomes
"Strategic Agility"   — competitive advantage, innovation speed
"Human Capital"       — talent retention, upskilling, workforce development
"Compliance & Risk"   — security posture, regulatory compliance, governance

Efficiency = same work done faster. Productivity = more work with same
resources. Assign both when story clearly claims both.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
B2. roi_evidence_sentences  (list of 2–3 verbatim strings)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Extract 2–3 verbatim sentences capturing the customer's ROI from AI.
Aim for: one sentence with a specific number + one with qualitative impact.
DO NOT extract: the vendor/ISV's own business growth metrics, generic
marketing claims, or sentences that only describe product features.
Minimum 1 sentence if fewer strong ROI sentences exist.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
B3. roi_numerical  (list of strings OR null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Extract EVERY specific number, percentage, dollar amount, time saving,
or quantified outcome from Story Content AND Executive Summary.
Always include unit and context: "40% cost reduction" not "40%".
DO NOT include vague claims. Return null if no specific figures exist.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C1. integration_framing  (string: "Strategic" or "Tactical")
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Read BOTH Story Content and Executive Summary.

Strategic: company describes AI as core long-term infrastructure,
foundational platform, or structural dependency. Language of
transformation, irreversibility, or mission-criticality.
Signals: "we rely on", "built on Azure", "our AI foundation",
"mission-critical", "long-term platform", "transformed how we operate",
"core to our strategy", "cannot operate without".

Tactical: productivity tool, efficiency gain, vendor feature.
Signals: "we use it for X tasks", "saves time on", "helps our team",
"productivity boost", "we chose Microsoft because".

DEFAULT: Tactical. Assign Strategic ONLY when EXPLICIT structural
dependency or transformation language is present.
Positive sentiment alone is NOT Strategic.
Using the word "strategic" to describe a decision or technology is NOT
sufficient — Strategic requires language of actual DEPENDENCY or
IRREVERSIBILITY, not just that it was a deliberate choice.
Competitive urgency ("we'll lose ground") is NOT sufficient.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C1a. framing_evidence_sentence  (1 verbatim sentence)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MUST be a direct quote from a named person OR a sentence from the
opening/closing paragraph explicitly stating the company's relationship
with Microsoft AI.

PRIORITY ORDER:
1. Direct executive quote containing framing language
2. Direct employee quote containing framing language
3. Opening/closing paragraph sentence about the company's AI relationship
4. Only if none exist: narrator sentence with explicit framing language

DO NOT use narrator descriptions, product feature sentences, or sentences
that use the word "strategic" without expressing actual dependency.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C2. security_priority_level  (integer 1, 2, 3, or null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
null: no mention of security, privacy, compliance, or data governance.
1:    mentioned briefly in passing — one sentence, not a driver.
      Example: "We trust Microsoft's security infrastructure."
2:    described as a key requirement or explicit adoption decision factor.
      Example: "Security and compliance were non-negotiable before rollout."
3:    core architectural foundation — built into the system design with
      specific certifications, zero-trust, or governance frameworks that
      the CUSTOMER designed or enforces (not just Microsoft's defaults).
      Example: "Zero-trust architecture with SOC2 — security is the
      system design."
DEFAULT when in doubt between 2 and 3: always choose 2.
If the security language describes Microsoft's built-in product security
rather than architecture the CUSTOMER designed, assign Level 1 or 2.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C2a. security_evidence_sentence  (1 verbatim sentence OR null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The single verbatim sentence that best justifies security_priority_level.
Required whenever security_priority_level is not null.
Return null only when security_priority_level = null.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
D1. primary_user_persona  (list — pick ALL that apply, exact strings)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"Frontline Workers"   — nurses, factory workers, field technicians,
                        retail staff, physical/operational roles
"Knowledge Workers"   — analysts, managers, lawyers, office staff
"Professional Devs"   — software engineers, IT teams, data scientists
"Executives"          — CEO, CTO, CIO, CFO, VP-level
"Students"            — learners in academic or corporate training

Assign Frontline Workers ONLY if physical/operational roles are
explicitly described as AI users, not just beneficiaries.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
D2. key_quote  (dict {quote, speaker, title} OR null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The single most relevant verbatim quote from any named person —
regardless of seniority. Choose the quote that most directly captures
how the AI is actually used OR why the company adopted it.
Do NOT default to the most senior person.
Must include: full name, title/role. Return null if no named quotes.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
D3. technical_summary  (string — exactly 2 sentences, synthesised)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ONLY field where you synthesise (not extract verbatim).
Sentence 1: what was built/deployed + which core AI product.
Sentence 2: what system/data it connects to + how it operates.
No marketing language.

=================================================================
EVIDENCE FIELDS — all _evidence_sentence and _signal_sentences
must be VERBATIM from the story. No paraphrasing.
=================================================================

=================================================================
CODING_NOTES — add a note only if:
- Override rule applied (Copilot Studio + ERP → L3)
- arch_level = 1 but agency_type = Agent
- Story is ISV/vendor, not end-customer
- Genuine ambiguity requiring a judgement call
- Story content too thin to code confidently
Otherwise return null.
=================================================================
""".strip()

# ── USER PROMPT (per story) ───────────────────────────────────────────────────
def build_user_prompt(company, products, story, executive_summary):
    return f"""Code this Microsoft customer story using the codebook in your system prompt.
Return JSON only — no explanation, no markdown.

COMPANY: {company}

PRODUCTS USED (for product signal check):
{products}

STORY CONTENT:
{story}

EXECUTIVE SUMMARY:
{executive_summary}

Return this exact JSON structure:
{{
  "arch_level": <1, 2, or 3>,
  "arch_level_signal_sentences": ["<verbatim>", "<verbatim>"],
  "agency_type": "<Tool or Agent>",
  "agency_evidence_sentence": "<verbatim>",
  "data_connectivity": "<General, Internal, or System>",
  "connectivity_evidence_sentence": "<verbatim>",
  "roi_metric": ["<value1>", "<value2>"],
  "roi_evidence_sentences": ["<verbatim 1>", "<verbatim 2>"],
  "roi_numerical": ["<figure with context>"] or null,
  "integration_framing": "<Strategic or Tactical>",
  "framing_evidence_sentence": "<verbatim direct quote>",
  "security_priority_level": <1, 2, 3, or null>,
  "security_evidence_sentence": "<verbatim>" or null,
  "primary_user_persona": ["<value1>"],
  "key_quote": {{"quote": "<verbatim>", "speaker": "<full name>", "title": "<role, company>"}} or null,
  "technical_summary": "<sentence 1>. <sentence 2>.",
  "coding_notes": "<note>" or null
}}"""

# ── VALIDATION ────────────────────────────────────────────────────────────────
VALID_ROI = {
    'Efficiency','Cost','Productivity','Quality','Employee Experience',
    'Customer Experience','Strategic Agility','Human Capital','Compliance & Risk'
}
VALID_PERSONA = {
    'Frontline Workers','Knowledge Workers','Professional Devs','Executives','Students'
}

def validate(result, idx, company):
    issues = []
    if result.get('arch_level') not in [1, 2, 3]:
        issues.append('arch_level invalid')
    if not result.get('arch_level_signal_sentences'):
        issues.append('arch_level_signal_sentences empty')
    if result.get('agency_type') not in ['Tool', 'Agent']:
        issues.append('agency_type invalid')
    if result.get('data_connectivity') not in ['General', 'Internal', 'System']:
        issues.append('data_connectivity invalid')
    if result.get('integration_framing') not in ['Strategic', 'Tactical']:
        issues.append('integration_framing invalid')
    if result.get('security_priority_level') not in [1, 2, 3, None]:
        issues.append('security_priority_level invalid')
    if result.get('security_priority_level') and not result.get('security_evidence_sentence'):
        issues.append('security_evidence_sentence missing (required when level not null)')
    for m in (result.get('roi_metric') or []):
        if m not in VALID_ROI:
            issues.append(f'roi_metric invalid: {m}')
    for p in (result.get('primary_user_persona') or []):
        if p not in VALID_PERSONA:
            issues.append(f'primary_user_persona invalid: {p}')
    if issues:
        print(f'  ⚠ Story {idx} ({company}): {"; ".join(issues)}')
    return issues

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
def load_stories():
    stories = []
    with open(INPUT_FILE, encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            if row.get('Customer Name', '').strip():
                stories.append(row)
    print(f'Loaded {len(stories)} stories from pipeline_input_900.csv')
    return stories

# ── SINGLE API CALL ───────────────────────────────────────────────────────────
def code_story(client, row, idx):
    user_prompt = build_user_prompt(
        company           = row.get('Customer Name', '').strip(),
        products          = row.get('Products Used', '').strip(),
        story             = row.get('Story Content', '').strip(),
        executive_summary = row.get('Executive summary', '').strip(),
    )
    response = client.messages.create(
        model      = MODEL,
        max_tokens = MAX_TOKENS,
        temperature= TEMPERATURE,
        system     = [
            {
                "type": "text",
                "text": SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"}  # caches codebook — saves ~80% input cost
            }
        ],
        messages   = [{"role": "user", "content": user_prompt}]
    )
    raw = response.content[0].text.strip()
    # Strip markdown fences if model adds them despite instructions
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
        raw = raw.strip()
    return json.loads(raw)

# ── MAIN PIPELINE ─────────────────────────────────────────────────────────────
def run_pipeline(start_from=0, limit=None):
    """
    start_from : story index to start/resume from
    limit      : max stories to process this run (None = all remaining)

    The pipeline writes each result immediately to OUTPUT_FILE (jsonl).
    If interrupted, resume by calling run_pipeline(start_from=N) —
    it skips stories already in the output file automatically.
    """
    client  = anthropic.Anthropic()
    stories = load_stories()

    # Find already-coded stories for resume logic
    coded = set()
    if Path(OUTPUT_FILE).exists():
        with open(OUTPUT_FILE) as f:
            for line in f:
                try:
                    coded.add(json.loads(line).get('_meta', {}).get('company_name', ''))
                except: pass
        if coded:
            print(f'Resuming — {len(coded)} stories already coded, skipping them')

    to_process = stories[start_from:]
    if limit: to_process = to_process[:limit]

    success = error = 0

    for i, row in enumerate(to_process):
        idx     = start_from + i
        company = row.get('Customer Name', '').strip()

        if company in coded:
            continue

        try:
            print(f'[{idx+1}/{len(stories)}] {company}')
            result = code_story(client, row, idx)
            issues = validate(result, idx, company)

            result['_meta'] = {
                'story_index'      : idx,
                'company_name'     : company,
                'validation_issues': issues,
            }

            with open(OUTPUT_FILE, 'a', encoding='utf-8') as f:
                f.write(json.dumps(result, ensure_ascii=False) + '\n')

            success += 1
            time.sleep(SLEEP)

        except json.JSONDecodeError as e:
            print(f'  ✗ JSON error: {e}')
            _log_error(idx, company, 'json_parse_error', str(e))
            error += 1

        except anthropic.RateLimitError:
            print(f'  Rate limit — sleeping 60s then retrying...')
            time.sleep(60)
            try:
                result = code_story(client, row, idx)
                result['_meta'] = {'story_index': idx, 'company_name': company}
                with open(OUTPUT_FILE, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(result, ensure_ascii=False) + '\n')
                success += 1
            except Exception as e2:
                _log_error(idx, company, 'retry_failed', str(e2))
                error += 1

        except Exception as e:
            print(f'  ✗ Unexpected error: {e}')
            _log_error(idx, company, 'unexpected', str(e))
            error += 1

    print(f'\n── Done: {success} coded · {error} errors ──')
    if error: print(f'   Error log: {ERROR_FILE}')

def _log_error(idx, company, kind, detail):
    with open(ERROR_FILE, 'a', encoding='utf-8') as f:
        f.write(json.dumps({
            'story_index': idx, 'company_name': company,
            'error': kind, 'detail': detail
        }) + '\n')

# ── LOAD RESULTS ──────────────────────────────────────────────────────────────
def load_results():
    results = []
    with open(OUTPUT_FILE, encoding='utf-8') as f:
        for line in f:
            try: results.append(json.loads(line))
            except: pass
    print(f'Loaded {len(results)} coded stories')
    return results

# ── TEST RUN ──────────────────────────────────────────────────────────────────
def run_test(n=5):
    """Always run this first. Inspect output before the full pipeline."""
    print(f'=== TEST RUN — first {n} stories ===\n')
    run_pipeline(start_from=0, limit=n)
    print('\n=== OUTPUT SAMPLE ===')
    results = load_results()
    for r in results[:n]:
        m = r.get('_meta', {})
        print(f"\n{m.get('company_name','?')}")
        print(f"  arch_level       : {r.get('arch_level')} ({r.get('agency_type')})")
        print(f"  data_connectivity: {r.get('data_connectivity')}")
        print(f"  framing          : {r.get('integration_framing')}")
        print(f"  security         : {r.get('security_priority_level')}")
        print(f"  roi_metric       : {r.get('roi_metric')}")
        print(f"  signal_sentences : {r.get('arch_level_signal_sentences')}")
        print(f"  key_quote        : {r.get('key_quote',{}).get('speaker','null')}")
        print(f"  coding_notes     : {r.get('coding_notes')}")
        if m.get('validation_issues'):
            print(f"  ⚠ ISSUES: {m['validation_issues']}")

# ── QUICK STATUS CHECK ────────────────────────────────────────────────────────
def check_status():
    coded  = sum(1 for _ in open(OUTPUT_FILE)) if Path(OUTPUT_FILE).exists() else 0
    errors = sum(1 for _ in open(ERROR_FILE))  if Path(ERROR_FILE).exists()  else 0
    print(f'Coded: {coded} / 900  |  Errors: {errors}')
    if errors:
        with open(ERROR_FILE) as f:
            for line in f:
                try:
                    e = json.loads(line)
                    print(f"  ✗ {e['company_name']}: {e['error']}")
                except: pass

# =============================================================================
# HOW TO RUN
# =============================================================================
# Step 1 — install SDK (first time only):
#   !pip install anthropic --break-system-packages
#
# Step 2 — set API key:
#   import os
#   os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
#
# Step 3 — always test first (5 stories, ~$0.05):
#   run_test(n=5)
#
# Step 4 — inspect output carefully, then run full pipeline:
#   run_pipeline()
#
# Step 5 — if Colab disconnects, resume from last position:
#   check_status()           # see how many coded so far
#   run_pipeline(start_from=450)   # resumes, skips already coded
#
# Step 6 — load all results when done:
#   results = load_results()
# =============================================================================

In [11]:
# Reference only — not executed (would call the API).
# run_test(n=5)

In [12]:
# Reference only — not executed.
# results = load_results()
# for r in results:
#     ...

#### Pipeline run (already completed — do not re-run)

In [13]:
# Reference only — already run; results saved to master_dataset_900.csv.
# run_pipeline()      # ~45 min, ~$18
# check_status()      # confirm 900/900

In [14]:
%%script false --no-raise-error
# (Reference only — this cell is not executed. The pipeline already ran; results are in master_dataset_900.csv.)
"""
=============================================================
BLOCK 9 — REPAIR CELL
Recodes the 100 stories that were either:
  - Skipped due to duplicate company name bug (24 stories)
  - Failed with JSON parse errors (76 stories)

HOW TO RUN:
  1. Make sure llm_coded_results.jsonl and llm_coding_errors.jsonl
     are in the same folder as pipeline_input_900.csv
  2. Run repair_test(n=5) first to inspect output
  3. Then run repair_pipeline() for all 100
=============================================================
"""

import anthropic, json, csv, time, os
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL        = "claude-sonnet-4-6"
TEMPERATURE  = 0
MAX_TOKENS   = 2000
INPUT_FILE   = '/content/drive/MyDrive/Microsoft900cases/pipeline_input_900.csv'
OUTPUT_FILE  = '/content/drive/MyDrive/Microsoft900cases/llm_coded_results.jsonl'
ERROR_FILE   = '/content/drive/MyDrive/Microsoft900cases/llm_coding_errors.jsonl'
REPAIR_ERROR_FILE = '/content/drive/MyDrive/Microsoft900cases/llm_repair_errors.jsonl'
SLEEP        = 0.5

# ── INDICES TO RECODE ─────────────────────────────────────────────────────────
REPAIR_INDICES = [
    23, 37, 40, 51, 55, 64, 70, 75, 88, 94, 97, 105, 107, 128, 157, 169,
    179, 188, 205, 222, 224, 242, 252, 273, 285, 297, 309, 318, 319, 322,
    333, 344, 348, 363, 368, 378, 385, 390, 391, 400, 404, 407, 409, 413,
    425, 429, 430, 431, 432, 440, 444, 445, 461, 462, 467, 477, 481, 484,
    490, 501, 525, 526, 528, 530, 532, 545, 553, 558, 563, 567, 572, 583,
    584, 586, 596, 607, 634, 659, 669, 674, 689, 700, 719, 731, 747, 754,
    764, 770, 771, 775, 812, 816, 831, 833, 835, 853, 855, 867, 869, 889
]

# ── SYSTEM PROMPT (v3 — with Student Experience, Sustainability, Security & Compliance added) ──
SYSTEM_PROMPT = """
You are a precise research coder for an academic study on enterprise AI adoption.
Your job: read Microsoft customer stories and extract structured data exactly
as defined in the codebook below. Follow every rule exactly.

CORE RULES (apply to everything):
- Never invent or infer information not present in the story.
- Extract evidence sentences VERBATIM — word-for-word, no paraphrasing, no shortening.
- When in doubt between two values, always choose the LOWER / more conservative option.
- Return ONLY valid JSON. No explanation, no preamble, no markdown fences.
- CRITICAL JSON RULE: All string values must be valid JSON strings. Any double quote
  characters that appear inside a string value MUST be escaped as \\". Any backslash
  characters MUST be escaped as \\\\. This applies especially to verbatim sentences
  which may contain quotes from the original text.

=================================================================
CODEBOOK — VARIABLE DEFINITIONS
=================================================================

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A1. arch_level  (integer: 1, 2, or 3)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Measures how deeply AI is integrated into the customer's operations.
CRITICAL: Do NOT assign based on product name alone.
You MUST verify the usage signal in the story content.

STEP 1 — Check for Level 3 product signal (check FIRST, overrides all):
Any of these products present → candidate for Level 3:
Azure AI Foundry, Azure AI Foundry Models, Azure AI Foundry Agent Service,
Azure OpenAI Service, Azure OpenAI, Azure Open AI Service,
Azure Machine Learning, Azure Machine Learning studio,
Azure AI Services, Azure AI Studio, Azure AI and Machine Learning,
Azure AI Search, Azure AI Document Intelligence, Azure AI Speech,
Azure AI Language, Azure AI Vision, Azure AI Bot Service,
Azure AI Content Safety, Azure AI Translator, Azure AI Video Indexer,
Azure AI Content Understanding, Azure AI Agent Service, Azure AI Service,
Azure AI Immersive Reader, Azure AI Metrics Advisor, Azure AI Model Catalog,
Azure AI and Machine Learning (AI), Azure OpenAI in Foundry Models,
Azure Document Intelligence in Foundry Tools, Azure Speech in Foundry Tools,
Microsoft Foundry, Foundry Models, Foundry Tools, Agents, Azure Phi,
Microsoft Dragon Copilot, Azure Open Datasets,
Microsoft Azure Data Manager for Agriculture

STEP 2 — Check for Level 2 product signal (only if no Level 3 present):
Any of these present → candidate for Level 2:
Microsoft Copilot Studio, Power Apps, Power Automate,
Microsoft Power Platform, Power Pages, Dataverse, Dataverse for Teams

STEP 3 — Default to Level 1 if no L3 or L2 signal present.
Neutral products (do NOT trigger any level):
M365 Copilot, GitHub Copilot, Microsoft Copilot, all Copilot-for-[Product]
variants, Azure, Teams, Power BI, SharePoint, Microsoft 365, Dynamics 365,
GitHub, Azure SQL, Azure Cosmos DB, Azure Blob Storage, Azure Functions,
Azure Container Apps, Azure Data Factory, Azure Logic Apps, Azure Kubernetes,
Azure Databricks, Azure App Service, Azure Virtual Machines, Microsoft Entra,
Microsoft Sentinel, Microsoft Purview, Microsoft Fabric, and all other
pure infrastructure or productivity products.

STEP 4 — Verify usage signal in story content:
LEVEL 3 confirmed if story describes: AI running autonomously, AI connected
to ERP/EHR/financial ledger/supply chain/core operational systems, customer
building OR deploying a Microsoft AI product deeply integrated into core
operational architecture.
NOTE: Pre-built Microsoft AI products (like Dragon Copilot) embedded
directly into EHR/ERP systems qualify as Level 3. The distinction is
between deep operational integration vs surface-level tool use.
LEVEL 2 confirmed if story describes: AI orchestrating multi-step workflows,
connecting to internal data (SharePoint, SQL, internal KB, CRM records),
human triggers overall process but AI executes steps within it.
LEVEL 1 confirmed if story describes: user manually invoking AI for single
discrete tasks (drafting, summarising, suggesting). No workflow orchestration.
No connection to core systems.

OVERRIDE RULE (always apply when reading story content):
Copilot Studio + ERP/EHR + autonomous operation described in story
→ assign Level 3 even if no Foundry/OpenAI product listed.

AMBIGUITY DEFAULTS:
- Between Level 1 and 2: assign Level 1 unless multi-step orchestration
  AND internal data connection are both clearly described.
- Between Level 2 and 3: assign Level 2 unless autonomous operation on
  CORE operational systems is clearly described.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A2. arch_level_signal_sentences  (list of 1–3 verbatim strings)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Extract 1–3 verbatim sentences from Story Content that PROVE the
arch_level assignment. Must show actual AI behaviour, system
integration, or operational usage.

DO extract: sentences describing autonomous AI operation, ERP/EHR
integration, custom AI construction, multi-step orchestration with
internal data, or explicit user-invoked single tasks.
DO NOT extract:
- Product name mentions alone ("The company uses Azure AI Foundry")
- Generic marketing claims ("AI transforms their business")
- Sentences from Executive Summary
- Sentences about non-AI tools unless that IS the primary AI deployment

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A3. agency_type  (string: "Tool" or "Agent")
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Tool: human explicitly invokes AI each time. Single discrete request.
AI produces output; human decides next action.
Signals: "I ask it to", "when I need it", "helps me write",
ambient recording that user activates.

Agent: AI acts proactively, executes multi-step tasks without
step-by-step human instruction, or makes autonomous decisions.
Signals: "automatically processes", "resolves tickets without human
intervention", "proactively flags", "orchestrates", "triggers when",
runs on schedule, multi-agent pipeline.

Counter-signals (do NOT assign Agent if only these):
"Copilot suggests the next step" (suggestion != autonomous action),
"we can ask it to do X" (still human-initiated),
automated batch processing pipelines where content arrives and the
system processes it automatically — this is automation, not agency.
Agent requires the AI to proactively DECIDE when and how to act,
not just run when triggered by data arrival or a scheduled job.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A4. data_connectivity  (string: "General", "Internal", or "System")
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Classify based on the PRIMARY AI deployment described in the story.

General: public knowledge only, no internal company data mentioned.

Internal: connects to company-specific INFORMATIONAL data —
  SharePoint, internal KB, email/Teams history, CRM contact records,
  pricing pages, intranet, internal wiki, Microsoft 365 data
  (meetings, emails, files, chats), security monitoring tools,
  SIEM platforms (Sentinel, Defender).

System: connects to TRANSACTIONAL/OPERATIONAL systems that actively
  run core business processes — ERP (SAP, Oracle), EHR (Epic, Cerner),
  financial ledger/accounting, supply chain management,
  HR/payroll (ADP), billing infrastructure, inventory management.

IMPORTANT:
- Cloud infrastructure provisioned at deploy time (Azure Container Apps,
  Azure Compute, databases) = NOT System. Platform resources are not
  enterprise operational systems.
- Security tools (Sentinel, Defender, SIEM, XDR) = Internal, not System.
- Microsoft 365, Teams, email, meetings, chats = Internal.
- CRM = Internal if contact records only; System only if it triggers
  automated operational transactions.
- If a story has multiple AI deployments, classify on the PRIMARY one.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
B1. roi_metric  (list — pick ALL that apply, exact strings only)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Use ONLY these exact strings — no variations:
"Efficiency"          — faster processes, time saved, automation
"Cost"                — money saved, cost reduction, consolidation savings
"Productivity"        — more output per person, higher throughput
"Quality"             — fewer errors, better accuracy, improved outcomes
"Employee Experience" — satisfaction, reduced burnout, meaningful work
"Customer Experience" — better service, faster response, improved outcomes
"Student Experience"  — improved learning outcomes, student engagement,
                        educational access, academic performance
"Strategic Agility"   — competitive advantage, innovation speed
"Human Capital"       — talent retention, upskilling, workforce development
"Compliance & Risk"   — security posture, regulatory compliance, governance
"Sustainability"      — environmental impact, carbon reduction, ESG outcomes
"Security & Compliance" — dedicated security architecture, zero-trust,
                          SOC2, data protection as primary outcome

Efficiency = same work done faster. Productivity = more work with same
resources. Assign both when story clearly claims both.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
B2. roi_evidence_sentences  (list of 2–3 verbatim strings)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Extract 2–3 verbatim sentences capturing the customer's ROI from AI.
Aim for: one sentence with a specific number + one with qualitative impact.
DO NOT extract: the vendor/ISV's own business growth metrics, generic
marketing claims, or sentences that only describe product features.
Minimum 1 sentence if fewer strong ROI sentences exist.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
B3. roi_numerical  (list of strings OR null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Extract EVERY specific number, percentage, dollar amount, time saving,
or quantified outcome from Story Content AND Executive Summary.
Always include unit and context: "40% cost reduction" not "40%".
DO NOT include vague claims. Return null if no specific figures exist.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C1. integration_framing  (string: "Strategic" or "Tactical")
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Read BOTH Story Content and Executive Summary.

Strategic: company describes AI as core long-term infrastructure,
foundational platform, or structural dependency. Language of
transformation, irreversibility, or mission-criticality.
Signals: "we rely on", "built on Azure", "our AI foundation",
"mission-critical", "long-term platform", "transformed how we operate",
"core to our strategy", "cannot operate without".

Tactical: productivity tool, efficiency gain, vendor feature.
Signals: "we use it for X tasks", "saves time on", "helps our team",
"productivity boost", "we chose Microsoft because".

DEFAULT: Tactical. Assign Strategic ONLY when EXPLICIT structural
dependency or transformation language is present.
Positive sentiment alone is NOT Strategic.
Using the word "strategic" to describe a decision or technology is NOT
sufficient — Strategic requires language of actual DEPENDENCY or
IRREVERSIBILITY, not just that it was a deliberate choice.
Competitive urgency ("we'll lose ground") is NOT sufficient.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C1a. framing_evidence_sentence  (1 verbatim sentence)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MUST be a direct quote from a named person OR a sentence from the
opening/closing paragraph explicitly stating the company's relationship
with Microsoft AI.

PRIORITY ORDER:
1. Direct executive quote containing framing language
2. Direct employee quote containing framing language
3. Opening/closing paragraph sentence about the company's AI relationship
4. Only if none exist: narrator sentence with explicit framing language

DO NOT use narrator descriptions, product feature sentences, or sentences
that use the word "strategic" without expressing actual dependency.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C2. security_priority_level  (integer 1, 2, 3, or null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
null: no mention of security, privacy, compliance, or data governance.
1:    mentioned briefly in passing — one sentence, not a driver.
      Example: "We trust Microsoft's security infrastructure."
2:    described as a key requirement or explicit adoption decision factor.
      Example: "Security and compliance were non-negotiable before rollout."
3:    core architectural foundation — built into the system design with
      specific certifications, zero-trust, or governance frameworks that
      the CUSTOMER designed or enforces (not just Microsoft's defaults).
      Example: "Zero-trust architecture with SOC2 — security is the
      system design."
DEFAULT when in doubt between 2 and 3: always choose 2.
If the security language describes Microsoft's built-in product security
rather than architecture the CUSTOMER designed, assign Level 1 or 2.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C2a. security_evidence_sentence  (1 verbatim sentence OR null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The single verbatim sentence that best justifies security_priority_level.
Required whenever security_priority_level is not null.
Return null only when security_priority_level = null.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
D1. primary_user_persona  (list — pick ALL that apply, exact strings)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"Frontline Workers"   — nurses, factory workers, field technicians,
                        retail staff, physical/operational roles
"Knowledge Workers"   — analysts, managers, lawyers, office staff
"Professional Devs"   — software engineers, IT teams, data scientists
"Executives"          — CEO, CTO, CIO, CFO, VP-level
"Students"            — learners in academic or corporate training

Assign Frontline Workers ONLY if physical/operational roles are
explicitly described as AI users, not just beneficiaries.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
D2. key_quote  (dict {quote, speaker, title} OR null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The single most relevant verbatim quote from any named person —
regardless of seniority. Choose the quote that most directly captures
how the AI is actually used OR why the company adopted it.
Do NOT default to the most senior person.
Must include: full name, title/role. Return null if no named quotes.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
D3. technical_summary  (string — exactly 2 sentences, synthesised)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ONLY field where you synthesise (not extract verbatim).
Sentence 1: what was built/deployed + which core AI product.
Sentence 2: what system/data it connects to + how it operates.
No marketing language.

=================================================================
EVIDENCE FIELDS — all _evidence_sentence and _signal_sentences
must be VERBATIM from the story. No paraphrasing.
=================================================================

=================================================================
CODING_NOTES — add a note only if:
- Override rule applied (Copilot Studio + ERP → L3)
- arch_level = 1 but agency_type = Agent
- Story is ISV/vendor, not end-customer
- Genuine ambiguity requiring a judgement call
- Story content too thin to code confidently
Otherwise return null.
=================================================================
""".strip()

# ── SMART QUOTE CLEANER ───────────────────────────────────────────────────────
def clean_text(text):
    """Replace smart/curly quotes with plain single quotes.
    This is the root cause of all JSON parse errors — the original Microsoft
    story text contains Unicode smart quotes (\u201c \u201d \u2018 \u2019).
    When the model copies these verbatim into JSON string values, it sometimes
    converts them to regular double quotes, breaking the JSON.
    Replacing them with single quotes in the INPUT is safe and guaranteed."""
    return (text
        .replace('\u201c', "'")   # left double smart quote  "
        .replace('\u201d', "'")   # right double smart quote "
        .replace('\u2018', "'")   # left single smart quote  '
        .replace('\u2019', "'")   # right single smart quote / apostrophe '
    )

# ── USER PROMPT ───────────────────────────────────────────────────────────────
def build_user_prompt(company, products, story, executive_summary):
    # Clean smart quotes from all text fields before sending
    story             = clean_text(story)
    executive_summary = clean_text(executive_summary)
    products          = clean_text(products)

    return f"""Code this Microsoft customer story using the codebook in your system prompt.
Return JSON only — no explanation, no markdown.

COMPANY: {company}

PRODUCTS USED (for product signal check):
{products}

STORY CONTENT:
{story}

EXECUTIVE SUMMARY:
{executive_summary}

Return this exact JSON structure:
{{
  "arch_level": <1, 2, or 3>,
  "arch_level_signal_sentences": ["<verbatim>", "<verbatim>"],
  "agency_type": "<Tool or Agent>",
  "agency_evidence_sentence": "<verbatim>",
  "data_connectivity": "<General, Internal, or System>",
  "connectivity_evidence_sentence": "<verbatim>",
  "roi_metric": ["<value1>", "<value2>"],
  "roi_evidence_sentences": ["<verbatim 1>", "<verbatim 2>"],
  "roi_numerical": ["<figure with context>"] or null,
  "integration_framing": "<Strategic or Tactical>",
  "framing_evidence_sentence": "<verbatim direct quote>",
  "security_priority_level": <1, 2, 3, or null>,
  "security_evidence_sentence": "<verbatim>" or null,
  "primary_user_persona": ["<value1>"],
  "key_quote": {{"quote": "<verbatim>", "speaker": "<full name>", "title": "<role, company>"}} or null,
  "technical_summary": "<sentence 1>. <sentence 2>.",
  "coding_notes": "<note>" or null
}}"""

# ── VALIDATION ────────────────────────────────────────────────────────────────
VALID_ROI = {
    'Efficiency', 'Cost', 'Productivity', 'Quality', 'Employee Experience',
    'Customer Experience', 'Student Experience', 'Strategic Agility',
    'Human Capital', 'Compliance & Risk', 'Sustainability', 'Security & Compliance'
}
VALID_PERSONA = {
    'Frontline Workers', 'Knowledge Workers', 'Professional Devs', 'Executives', 'Students'
}

def validate(result, idx, company):
    issues = []
    if result.get('arch_level') not in [1, 2, 3]:
        issues.append('arch_level invalid')
    if not result.get('arch_level_signal_sentences'):
        issues.append('arch_level_signal_sentences empty')
    if result.get('agency_type') not in ['Tool', 'Agent']:
        issues.append('agency_type invalid')
    if result.get('data_connectivity') not in ['General', 'Internal', 'System']:
        issues.append('data_connectivity invalid')
    if result.get('integration_framing') not in ['Strategic', 'Tactical']:
        issues.append('integration_framing invalid')
    if result.get('security_priority_level') not in [1, 2, 3, None]:
        issues.append('security_priority_level invalid')
    if result.get('security_priority_level') and not result.get('security_evidence_sentence'):
        issues.append('security_evidence_sentence missing')
    for m in (result.get('roi_metric') or []):
        if m not in VALID_ROI:
            issues.append(f'roi_metric invalid: {m}')
    for p in (result.get('primary_user_persona') or []):
        if p not in VALID_PERSONA:
            issues.append(f'primary_user_persona invalid: {p}')
    if issues:
        print(f'  ⚠ Story {idx} ({company}): {"; ".join(issues)}')
    return issues

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
def load_stories():
    import csv
    stories = []
    with open(INPUT_FILE, encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            if row.get('Customer Name', '').strip():
                stories.append(row)
    print(f'Loaded {len(stories)} stories')
    return stories

# ── JSON CLEANING ─────────────────────────────────────────────────────────────
def clean_and_parse(raw):
    """Strip markdown fences then parse. Smart quotes are already cleaned
    from input so the model output should never contain unescaped quotes."""
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
        raw = raw.strip()
    return json.loads(raw)

# ── SINGLE API CALL ───────────────────────────────────────────────────────────
def code_story(client, row, idx):
    user_prompt = build_user_prompt(
        company           = row.get('Customer Name', '').strip(),
        products          = row.get('Products Used', '').strip(),
        story             = row.get('Story Content', '').strip(),
        executive_summary = row.get('Executive summary', '').strip(),
    )
    response = client.messages.create(
        model       = MODEL,
        max_tokens  = MAX_TOKENS,
        temperature = TEMPERATURE,
        system      = [{
            "type": "text",
            "text": SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"}
        }],
        messages    = [{"role": "user", "content": user_prompt}]
    )
    raw = response.content[0].text.strip()
    return clean_and_parse(raw)

# ── LOG ERROR ─────────────────────────────────────────────────────────────────
def _log_error(idx, company, kind, detail):
    with open(REPAIR_ERROR_FILE, 'a', encoding='utf-8') as f:
        f.write(json.dumps({
            'story_index': idx, 'company_name': company,
            'error': kind, 'detail': detail
        }) + '\n')

# ── REPAIR PIPELINE ───────────────────────────────────────────────────────────
def repair_pipeline(limit=None):
    """
    Recodes only the 100 stories that were missing or errored.
    Appends results to the existing OUTPUT_FILE.
    Already-repaired stories are skipped automatically on resume.

    limit: int — process only first N stories (useful for testing)
    """
    client  = anthropic.Anthropic()
    stories = load_stories()

    # Find which repair indices are already done (resume support)
    repaired = set()
    if Path(OUTPUT_FILE).exists():
        with open(OUTPUT_FILE, encoding='utf-8') as f:
            for line in f:
                try:
                    r = json.loads(line)
                    idx = r.get('_meta', {}).get('story_index')
                    if idx in REPAIR_INDICES:
                        repaired.add(idx)
                except: pass
    if repaired:
        print(f'Resuming repair — {len(repaired)} already done, skipping them')

    to_process = [i for i in REPAIR_INDICES if i not in repaired]
    if limit:
        to_process = to_process[:limit]

    print(f'Stories to recode this run: {len(to_process)}\n')
    success = error = 0

    for idx in to_process:
        company = stories[idx].get('Customer Name', '').strip()
        try:
            print(f'[{idx}] {company}')
            result  = code_story(client, stories[idx], idx)
            issues  = validate(result, idx, company)
            result['_meta'] = {
                'story_index'      : idx,
                'company_name'     : company,
                'validation_issues': issues,
                'repair_run'       : True,
            }
            with open(OUTPUT_FILE, 'a', encoding='utf-8') as f:
                f.write(json.dumps(result, ensure_ascii=False) + '\n')
            success += 1
            time.sleep(SLEEP)

        except json.JSONDecodeError as e:
            print(f'  ✗ JSON still broken after repair attempt: {e}')
            _log_error(idx, company, 'json_parse_error_after_repair', str(e))
            error += 1

        except anthropic.RateLimitError:
            print(f'  Rate limit — sleeping 60s then retrying...')
            time.sleep(60)
            try:
                result = code_story(client, stories[idx], idx)
                result['_meta'] = {
                    'story_index': idx, 'company_name': company, 'repair_run': True
                }
                with open(OUTPUT_FILE, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(result, ensure_ascii=False) + '\n')
                success += 1
            except Exception as e2:
                _log_error(idx, company, 'retry_failed', str(e2))
                error += 1

        except Exception as e:
            print(f'  ✗ Unexpected error: {e}')
            _log_error(idx, company, 'unexpected', str(e))
            error += 1

    print(f'\n── Repair done: {success} coded · {error} errors ──')
    if error:
        print(f'   Error log: {REPAIR_ERROR_FILE}')

# ── REPAIR STATUS ─────────────────────────────────────────────────────────────
def repair_status():
    done = set()
    if Path(OUTPUT_FILE).exists():
        with open(OUTPUT_FILE, encoding='utf-8') as f:
            for line in f:
                try:
                    r = json.loads(line)
                    idx = r.get('_meta', {}).get('story_index')
                    if idx in REPAIR_INDICES:
                        done.add(idx)
                except: pass
    remaining = [i for i in REPAIR_INDICES if i not in done]
    errors = 0
    if Path(REPAIR_ERROR_FILE).exists():
        errors = sum(1 for _ in open(REPAIR_ERROR_FILE))
    print(f'Repair progress: {len(done)}/100 done | {len(remaining)} remaining | {errors} errors')
    if remaining:
        print(f'Remaining indices: {remaining}')

# =============================================================================
# HOW TO RUN
# =============================================================================
# Step 1 — test first (5 stories, ~$0.05):
#   repair_pipeline(limit=5)
#
# Step 2 — check output looks good, then run all 100:
#   repair_pipeline()
#
# Step 3 — if Colab disconnects, just re-run repair_pipeline()
#   It will automatically skip already-repaired stories.
#
# Step 4 — check status anytime:
#   repair_status()
# =============================================================================

In [15]:
# Reference only — not executed.
# repair_pipeline(limit=5)
# repair_pipeline()

In [16]:
%%script false --no-raise-error
# (Reference only — this cell is not executed. The pipeline already ran; results are in master_dataset_900.csv.)
"""
=============================================================
BLOCK 9 — REPAIR CELL
Recodes the 100 stories that were either:
  - Skipped due to duplicate company name bug (24 stories)
  - Failed with JSON parse errors (76 stories)

HOW TO RUN:
  1. Make sure llm_coded_results.jsonl and llm_coding_errors.jsonl
     are in the same folder as pipeline_input_900.csv
  2. Run repair_test(n=5) first to inspect output
  3. Then run repair_pipeline() for all 100
=============================================================
"""

import anthropic, json, csv, time, os
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL        = "claude-sonnet-4-6"
TEMPERATURE  = 0
MAX_TOKENS   = 2000
INPUT_FILE   = '/content/drive/MyDrive/Microsoft900cases/pipeline_input_900.csv'
OUTPUT_FILE  = '/content/drive/MyDrive/Microsoft900cases/llm_coded_results.jsonl'
ERROR_FILE   = '/content/drive/MyDrive/Microsoft900cases/llm_coding_errors.jsonl'
REPAIR_ERROR_FILE = '/content/drive/MyDrive/Microsoft900cases/llm_repair_errors.jsonl'
SLEEP        = 0.5

# ── INDICES TO RECODE ─────────────────────────────────────────────────────────
# Full original list (used for resume tracking)
REPAIR_INDICES_FULL = [
    23, 37, 40, 51, 55, 64, 70, 75, 88, 94, 97, 105, 107, 128, 157, 169,
    179, 188, 205, 222, 224, 242, 252, 273, 285, 297, 309, 318, 319, 322,
    333, 344, 348, 363, 368, 378, 385, 390, 391, 400, 404, 407, 409, 413,
    425, 429, 430, 431, 432, 440, 444, 445, 461, 462, 467, 477, 481, 484,
    490, 501, 525, 526, 528, 530, 532, 545, 553, 558, 563, 567, 572, 583,
    584, 586, 596, 607, 634, 659, 669, 674, 689, 700, 719, 731, 747, 754,
    764, 770, 771, 775, 812, 816, 831, 833, 835, 853, 855, 867, 869, 889
]

# Only the 3 that still failed after the first repair run
REPAIR_INDICES = [407, 432, 530]

# ── SYSTEM PROMPT (v3 — with Student Experience, Sustainability, Security & Compliance added) ──
SYSTEM_PROMPT = """
You are a precise research coder for an academic study on enterprise AI adoption.
Your job: read Microsoft customer stories and extract structured data exactly
as defined in the codebook below. Follow every rule exactly.

CORE RULES (apply to everything):
- Never invent or infer information not present in the story.
- Extract evidence sentences VERBATIM — word-for-word, no paraphrasing, no shortening.
- When in doubt between two values, always choose the LOWER / more conservative option.
- Return ONLY valid JSON. No explanation, no preamble, no markdown fences.
- CRITICAL JSON RULE: All string values must be valid JSON strings. Any double quote
  characters that appear inside a string value MUST be escaped as \\". Any backslash
  characters MUST be escaped as \\\\. This applies especially to verbatim sentences
  which may contain quotes from the original text.

=================================================================
CODEBOOK — VARIABLE DEFINITIONS
=================================================================

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A1. arch_level  (integer: 1, 2, or 3)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Measures how deeply AI is integrated into the customer's operations.
CRITICAL: Do NOT assign based on product name alone.
You MUST verify the usage signal in the story content.

STEP 1 — Check for Level 3 product signal (check FIRST, overrides all):
Any of these products present → candidate for Level 3:
Azure AI Foundry, Azure AI Foundry Models, Azure AI Foundry Agent Service,
Azure OpenAI Service, Azure OpenAI, Azure Open AI Service,
Azure Machine Learning, Azure Machine Learning studio,
Azure AI Services, Azure AI Studio, Azure AI and Machine Learning,
Azure AI Search, Azure AI Document Intelligence, Azure AI Speech,
Azure AI Language, Azure AI Vision, Azure AI Bot Service,
Azure AI Content Safety, Azure AI Translator, Azure AI Video Indexer,
Azure AI Content Understanding, Azure AI Agent Service, Azure AI Service,
Azure AI Immersive Reader, Azure AI Metrics Advisor, Azure AI Model Catalog,
Azure AI and Machine Learning (AI), Azure OpenAI in Foundry Models,
Azure Document Intelligence in Foundry Tools, Azure Speech in Foundry Tools,
Microsoft Foundry, Foundry Models, Foundry Tools, Agents, Azure Phi,
Microsoft Dragon Copilot, Azure Open Datasets,
Microsoft Azure Data Manager for Agriculture

STEP 2 — Check for Level 2 product signal (only if no Level 3 present):
Any of these present → candidate for Level 2:
Microsoft Copilot Studio, Power Apps, Power Automate,
Microsoft Power Platform, Power Pages, Dataverse, Dataverse for Teams

STEP 3 — Default to Level 1 if no L3 or L2 signal present.
Neutral products (do NOT trigger any level):
M365 Copilot, GitHub Copilot, Microsoft Copilot, all Copilot-for-[Product]
variants, Azure, Teams, Power BI, SharePoint, Microsoft 365, Dynamics 365,
GitHub, Azure SQL, Azure Cosmos DB, Azure Blob Storage, Azure Functions,
Azure Container Apps, Azure Data Factory, Azure Logic Apps, Azure Kubernetes,
Azure Databricks, Azure App Service, Azure Virtual Machines, Microsoft Entra,
Microsoft Sentinel, Microsoft Purview, Microsoft Fabric, and all other
pure infrastructure or productivity products.

STEP 4 — Verify usage signal in story content:
LEVEL 3 confirmed if story describes: AI running autonomously, AI connected
to ERP/EHR/financial ledger/supply chain/core operational systems, customer
building OR deploying a Microsoft AI product deeply integrated into core
operational architecture.
NOTE: Pre-built Microsoft AI products (like Dragon Copilot) embedded
directly into EHR/ERP systems qualify as Level 3. The distinction is
between deep operational integration vs surface-level tool use.
LEVEL 2 confirmed if story describes: AI orchestrating multi-step workflows,
connecting to internal data (SharePoint, SQL, internal KB, CRM records),
human triggers overall process but AI executes steps within it.
LEVEL 1 confirmed if story describes: user manually invoking AI for single
discrete tasks (drafting, summarising, suggesting). No workflow orchestration.
No connection to core systems.

OVERRIDE RULE (always apply when reading story content):
Copilot Studio + ERP/EHR + autonomous operation described in story
→ assign Level 3 even if no Foundry/OpenAI product listed.

AMBIGUITY DEFAULTS:
- Between Level 1 and 2: assign Level 1 unless multi-step orchestration
  AND internal data connection are both clearly described.
- Between Level 2 and 3: assign Level 2 unless autonomous operation on
  CORE operational systems is clearly described.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A2. arch_level_signal_sentences  (list of 1–3 verbatim strings)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Extract 1–3 verbatim sentences from Story Content that PROVE the
arch_level assignment. Must show actual AI behaviour, system
integration, or operational usage.

DO extract: sentences describing autonomous AI operation, ERP/EHR
integration, custom AI construction, multi-step orchestration with
internal data, or explicit user-invoked single tasks.
DO NOT extract:
- Product name mentions alone ("The company uses Azure AI Foundry")
- Generic marketing claims ("AI transforms their business")
- Sentences from Executive Summary
- Sentences about non-AI tools unless that IS the primary AI deployment

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A3. agency_type  (string: "Tool" or "Agent")
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Tool: human explicitly invokes AI each time. Single discrete request.
AI produces output; human decides next action.
Signals: "I ask it to", "when I need it", "helps me write",
ambient recording that user activates.

Agent: AI acts proactively, executes multi-step tasks without
step-by-step human instruction, or makes autonomous decisions.
Signals: "automatically processes", "resolves tickets without human
intervention", "proactively flags", "orchestrates", "triggers when",
runs on schedule, multi-agent pipeline.

Counter-signals (do NOT assign Agent if only these):
"Copilot suggests the next step" (suggestion != autonomous action),
"we can ask it to do X" (still human-initiated),
automated batch processing pipelines where content arrives and the
system processes it automatically — this is automation, not agency.
Agent requires the AI to proactively DECIDE when and how to act,
not just run when triggered by data arrival or a scheduled job.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A4. data_connectivity  (string: "General", "Internal", or "System")
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Classify based on the PRIMARY AI deployment described in the story.

General: public knowledge only, no internal company data mentioned.

Internal: connects to company-specific INFORMATIONAL data —
  SharePoint, internal KB, email/Teams history, CRM contact records,
  pricing pages, intranet, internal wiki, Microsoft 365 data
  (meetings, emails, files, chats), security monitoring tools,
  SIEM platforms (Sentinel, Defender).

System: connects to TRANSACTIONAL/OPERATIONAL systems that actively
  run core business processes — ERP (SAP, Oracle), EHR (Epic, Cerner),
  financial ledger/accounting, supply chain management,
  HR/payroll (ADP), billing infrastructure, inventory management.

IMPORTANT:
- Cloud infrastructure provisioned at deploy time (Azure Container Apps,
  Azure Compute, databases) = NOT System. Platform resources are not
  enterprise operational systems.
- Security tools (Sentinel, Defender, SIEM, XDR) = Internal, not System.
- Microsoft 365, Teams, email, meetings, chats = Internal.
- CRM = Internal if contact records only; System only if it triggers
  automated operational transactions.
- If a story has multiple AI deployments, classify on the PRIMARY one.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
B1. roi_metric  (list — pick ALL that apply, exact strings only)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Use ONLY these exact strings — no variations:
"Efficiency"          — faster processes, time saved, automation
"Cost"                — money saved, cost reduction, consolidation savings
"Productivity"        — more output per person, higher throughput
"Quality"             — fewer errors, better accuracy, improved outcomes
"Employee Experience" — satisfaction, reduced burnout, meaningful work
"Customer Experience" — better service, faster response, improved outcomes
"Student Experience"  — improved learning outcomes, student engagement,
                        educational access, academic performance
"Strategic Agility"   — competitive advantage, innovation speed
"Human Capital"       — talent retention, upskilling, workforce development
"Compliance & Risk"   — security posture, regulatory compliance, governance
"Sustainability"      — environmental impact, carbon reduction, ESG outcomes
"Security & Compliance" — dedicated security architecture, zero-trust,
                          SOC2, data protection as primary outcome

Efficiency = same work done faster. Productivity = more work with same
resources. Assign both when story clearly claims both.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
B2. roi_evidence_sentences  (list of 2–3 verbatim strings)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Extract 2–3 verbatim sentences capturing the customer's ROI from AI.
Aim for: one sentence with a specific number + one with qualitative impact.
DO NOT extract: the vendor/ISV's own business growth metrics, generic
marketing claims, or sentences that only describe product features.
Minimum 1 sentence if fewer strong ROI sentences exist.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
B3. roi_numerical  (list of strings OR null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Extract EVERY specific number, percentage, dollar amount, time saving,
or quantified outcome from Story Content AND Executive Summary.
Always include unit and context: "40% cost reduction" not "40%".
DO NOT include vague claims. Return null if no specific figures exist.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C1. integration_framing  (string: "Strategic" or "Tactical")
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Read BOTH Story Content and Executive Summary.

Strategic: company describes AI as core long-term infrastructure,
foundational platform, or structural dependency. Language of
transformation, irreversibility, or mission-criticality.
Signals: "we rely on", "built on Azure", "our AI foundation",
"mission-critical", "long-term platform", "transformed how we operate",
"core to our strategy", "cannot operate without".

Tactical: productivity tool, efficiency gain, vendor feature.
Signals: "we use it for X tasks", "saves time on", "helps our team",
"productivity boost", "we chose Microsoft because".

DEFAULT: Tactical. Assign Strategic ONLY when EXPLICIT structural
dependency or transformation language is present.
Positive sentiment alone is NOT Strategic.
Using the word "strategic" to describe a decision or technology is NOT
sufficient — Strategic requires language of actual DEPENDENCY or
IRREVERSIBILITY, not just that it was a deliberate choice.
Competitive urgency ("we'll lose ground") is NOT sufficient.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C1a. framing_evidence_sentence  (1 verbatim sentence)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MUST be a direct quote from a named person OR a sentence from the
opening/closing paragraph explicitly stating the company's relationship
with Microsoft AI.

PRIORITY ORDER:
1. Direct executive quote containing framing language
2. Direct employee quote containing framing language
3. Opening/closing paragraph sentence about the company's AI relationship
4. Only if none exist: narrator sentence with explicit framing language

DO NOT use narrator descriptions, product feature sentences, or sentences
that use the word "strategic" without expressing actual dependency.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C2. security_priority_level  (integer 1, 2, 3, or null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
null: no mention of security, privacy, compliance, or data governance.
1:    mentioned briefly in passing — one sentence, not a driver.
      Example: "We trust Microsoft's security infrastructure."
2:    described as a key requirement or explicit adoption decision factor.
      Example: "Security and compliance were non-negotiable before rollout."
3:    core architectural foundation — built into the system design with
      specific certifications, zero-trust, or governance frameworks that
      the CUSTOMER designed or enforces (not just Microsoft's defaults).
      Example: "Zero-trust architecture with SOC2 — security is the
      system design."
DEFAULT when in doubt between 2 and 3: always choose 2.
If the security language describes Microsoft's built-in product security
rather than architecture the CUSTOMER designed, assign Level 1 or 2.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C2a. security_evidence_sentence  (1 verbatim sentence OR null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The single verbatim sentence that best justifies security_priority_level.
Required whenever security_priority_level is not null.
Return null only when security_priority_level = null.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
D1. primary_user_persona  (list — pick ALL that apply, exact strings)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"Frontline Workers"   — nurses, factory workers, field technicians,
                        retail staff, physical/operational roles
"Knowledge Workers"   — analysts, managers, lawyers, office staff
"Professional Devs"   — software engineers, IT teams, data scientists
"Executives"          — CEO, CTO, CIO, CFO, VP-level
"Students"            — learners in academic or corporate training

Assign Frontline Workers ONLY if physical/operational roles are
explicitly described as AI users, not just beneficiaries.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
D2. key_quote  (dict {quote, speaker, title} OR null)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The single most relevant verbatim quote from any named person —
regardless of seniority. Choose the quote that most directly captures
how the AI is actually used OR why the company adopted it.
Do NOT default to the most senior person.
Must include: full name, title/role. Return null if no named quotes.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
D3. technical_summary  (string — exactly 2 sentences, synthesised)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ONLY field where you synthesise (not extract verbatim).
Sentence 1: what was built/deployed + which core AI product.
Sentence 2: what system/data it connects to + how it operates.
No marketing language.

=================================================================
EVIDENCE FIELDS — all _evidence_sentence and _signal_sentences
must be VERBATIM from the story. No paraphrasing.
=================================================================

=================================================================
CODING_NOTES — add a note only if:
- Override rule applied (Copilot Studio + ERP → L3)
- arch_level = 1 but agency_type = Agent
- Story is ISV/vendor, not end-customer
- Genuine ambiguity requiring a judgement call
- Story content too thin to code confidently
Otherwise return null.
=================================================================
""".strip()

# ── SMART QUOTE CLEANER ───────────────────────────────────────────────────────
def clean_text(text):
    """Replace all quote characters with plain single quotes.
    Two sources of JSON parse errors found in story text:
    1. Smart/curly quotes (\u201c \u201d \u2018 \u2019) — Microsoft formatting
    2. Regular double quotes (") — appear in product names e.g. \'Fidal IA\'
    Both are replaced with single quotes, which are safe inside JSON strings."""
    return (str(text)
        .replace('\u201c', "'")   # left double smart quote  "
        .replace('\u201d', "'")   # right double smart quote "
        .replace('\u2018', "'")   # left single smart quote  '
        .replace('\u2019', "'")   # right single smart quote / apostrophe '
        .replace('"',      "'")   # regular double quote "
    )

# ── USER PROMPT ───────────────────────────────────────────────────────────────
def build_user_prompt(company, products, story, executive_summary):
    # Clean smart quotes from all text fields before sending
    story             = clean_text(story)
    executive_summary = clean_text(executive_summary)
    products          = clean_text(products)

    return f"""Code this Microsoft customer story using the codebook in your system prompt.
Return JSON only — no explanation, no markdown.

COMPANY: {company}

PRODUCTS USED (for product signal check):
{products}

STORY CONTENT:
{story}

EXECUTIVE SUMMARY:
{executive_summary}

Return this exact JSON structure:
{{
  "arch_level": <1, 2, or 3>,
  "arch_level_signal_sentences": ["<verbatim>", "<verbatim>"],
  "agency_type": "<Tool or Agent>",
  "agency_evidence_sentence": "<verbatim>",
  "data_connectivity": "<General, Internal, or System>",
  "connectivity_evidence_sentence": "<verbatim>",
  "roi_metric": ["<value1>", "<value2>"],
  "roi_evidence_sentences": ["<verbatim 1>", "<verbatim 2>"],
  "roi_numerical": ["<figure with context>"] or null,
  "integration_framing": "<Strategic or Tactical>",
  "framing_evidence_sentence": "<verbatim direct quote>",
  "security_priority_level": <1, 2, 3, or null>,
  "security_evidence_sentence": "<verbatim>" or null,
  "primary_user_persona": ["<value1>"],
  "key_quote": {{"quote": "<verbatim>", "speaker": "<full name>", "title": "<role, company>"}} or null,
  "technical_summary": "<sentence 1>. <sentence 2>.",
  "coding_notes": "<note>" or null
}}"""

# ── VALIDATION ────────────────────────────────────────────────────────────────
VALID_ROI = {
    'Efficiency', 'Cost', 'Productivity', 'Quality', 'Employee Experience',
    'Customer Experience', 'Student Experience', 'Strategic Agility',
    'Human Capital', 'Compliance & Risk', 'Sustainability', 'Security & Compliance',
    'Safety & Compliance', 'Patient Experience'
}
VALID_PERSONA = {
    'Frontline Workers', 'Knowledge Workers', 'Professional Devs', 'Executives', 'Students', 'Customers'
}

def validate(result, idx, company):
    issues = []
    if result.get('arch_level') not in [1, 2, 3]:
        issues.append('arch_level invalid')
    if not result.get('arch_level_signal_sentences'):
        issues.append('arch_level_signal_sentences empty')
    if result.get('agency_type') not in ['Tool', 'Agent']:
        issues.append('agency_type invalid')
    if result.get('data_connectivity') not in ['General', 'Internal', 'System']:
        issues.append('data_connectivity invalid')
    if result.get('integration_framing') not in ['Strategic', 'Tactical']:
        issues.append('integration_framing invalid')
    if result.get('security_priority_level') not in [1, 2, 3, None]:
        issues.append('security_priority_level invalid')
    if result.get('security_priority_level') and not result.get('security_evidence_sentence'):
        issues.append('security_evidence_sentence missing')
    for m in (result.get('roi_metric') or []):
        if m not in VALID_ROI:
            issues.append(f'roi_metric invalid: {m}')
    for p in (result.get('primary_user_persona') or []):
        if p not in VALID_PERSONA:
            issues.append(f'primary_user_persona invalid: {p}')
    if issues:
        print(f'  ⚠ Story {idx} ({company}): {"; ".join(issues)}')
    return issues

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
def load_stories():
    import csv
    stories = []
    with open(INPUT_FILE, encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            if row.get('Customer Name', '').strip():
                stories.append(row)
    print(f'Loaded {len(stories)} stories')
    return stories

# ── JSON CLEANING ─────────────────────────────────────────────────────────────
def clean_and_parse(raw):
    """Strip markdown fences then parse. Smart quotes are already cleaned
    from input so the model output should never contain unescaped quotes."""
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
        raw = raw.strip()
    return json.loads(raw)

# ── SINGLE API CALL ───────────────────────────────────────────────────────────
def code_story(client, row, idx):
    user_prompt = build_user_prompt(
        company           = row.get('Customer Name', '').strip(),
        products          = row.get('Products Used', '').strip(),
        story             = row.get('Story Content', '').strip(),
        executive_summary = row.get('Executive summary', '').strip(),
    )
    response = client.messages.create(
        model       = MODEL,
        max_tokens  = MAX_TOKENS,
        temperature = TEMPERATURE,
        system      = [{
            "type": "text",
            "text": SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"}
        }],
        messages    = [{"role": "user", "content": user_prompt}]
    )
    raw = response.content[0].text.strip()
    return clean_and_parse(raw)

# ── LOG ERROR ─────────────────────────────────────────────────────────────────
def _log_error(idx, company, kind, detail):
    with open(REPAIR_ERROR_FILE, 'a', encoding='utf-8') as f:
        f.write(json.dumps({
            'story_index': idx, 'company_name': company,
            'error': kind, 'detail': detail
        }) + '\n')

# ── REPAIR PIPELINE ───────────────────────────────────────────────────────────
def repair_pipeline(limit=None):
    """
    Recodes only the 100 stories that were missing or errored.
    Appends results to the existing OUTPUT_FILE.
    Already-repaired stories are skipped automatically on resume.

    limit: int — process only first N stories (useful for testing)
    """
    client  = anthropic.Anthropic()
    stories = load_stories()

    # Find which repair indices are already done (resume support)
    repaired = set()
    if Path(OUTPUT_FILE).exists():
        with open(OUTPUT_FILE, encoding='utf-8') as f:
            for line in f:
                try:
                    r = json.loads(line)
                    idx = r.get('_meta', {}).get('story_index')
                    if idx in REPAIR_INDICES:
                        repaired.add(idx)
                except: pass
    if repaired:
        print(f'Resuming repair — {len(repaired)} already done, skipping them')

    to_process = [i for i in REPAIR_INDICES if i not in repaired]
    if limit:
        to_process = to_process[:limit]

    print(f'Stories to recode this run: {len(to_process)}\n')
    success = error = 0

    for idx in to_process:
        company = stories[idx].get('Customer Name', '').strip()
        try:
            print(f'[{idx}] {company}')
            result  = code_story(client, stories[idx], idx)
            issues  = validate(result, idx, company)
            result['_meta'] = {
                'story_index'      : idx,
                'company_name'     : company,
                'validation_issues': issues,
                'repair_run'       : True,
            }
            with open(OUTPUT_FILE, 'a', encoding='utf-8') as f:
                f.write(json.dumps(result, ensure_ascii=False) + '\n')
            success += 1
            time.sleep(SLEEP)

        except json.JSONDecodeError as e:
            print(f'  ✗ JSON still broken after repair attempt: {e}')
            _log_error(idx, company, 'json_parse_error_after_repair', str(e))
            error += 1

        except anthropic.RateLimitError:
            print(f'  Rate limit — sleeping 60s then retrying...')
            time.sleep(60)
            try:
                result = code_story(client, stories[idx], idx)
                result['_meta'] = {
                    'story_index': idx, 'company_name': company, 'repair_run': True
                }
                with open(OUTPUT_FILE, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(result, ensure_ascii=False) + '\n')
                success += 1
            except Exception as e2:
                _log_error(idx, company, 'retry_failed', str(e2))
                error += 1

        except Exception as e:
            print(f'  ✗ Unexpected error: {e}')
            _log_error(idx, company, 'unexpected', str(e))
            error += 1

    print(f'\n── Repair done: {success} coded · {error} errors ──')
    if error:
        print(f'   Error log: {REPAIR_ERROR_FILE}')

# ── REPAIR STATUS ─────────────────────────────────────────────────────────────
def repair_status():
    done = set()
    if Path(OUTPUT_FILE).exists():
        with open(OUTPUT_FILE, encoding='utf-8') as f:
            for line in f:
                try:
                    r = json.loads(line)
                    idx = r.get('_meta', {}).get('story_index')
                    if idx in REPAIR_INDICES:
                        done.add(idx)
                except: pass
    remaining = [i for i in REPAIR_INDICES if i not in done]
    errors = 0
    if Path(REPAIR_ERROR_FILE).exists():
        errors = sum(1 for _ in open(REPAIR_ERROR_FILE))
    print(f'Repair progress: {len(done)}/100 done | {len(remaining)} remaining | {errors} errors')
    if remaining:
        print(f'Remaining indices: {remaining}')

# =============================================================================
# HOW TO RUN
# =============================================================================
# Step 1 — test first (5 stories, ~$0.05):
#   repair_pipeline(limit=5)
#
# Step 2 — check output looks good, then run all 100:
#   repair_pipeline()
#
# Step 3 — if Colab disconnects, just re-run repair_pipeline()
#   It will automatically skip already-repaired stories.
#
# Step 4 — check status anytime:
#   repair_status()
# =============================================================================

In [17]:
# Reference only — not executed.
# repair_pipeline()

#### Final merge (already completed)

In [18]:
# Reference only — already run; master_dataset_900.csv already exists.
# (Merge code retained below for transparency.)

In [19]:
"""
=============================================================
BLOCK 10 — FINAL MERGE
1. Deduplicates llm_coded_results.jsonl (keeps latest entry
   per story_index in case of any duplicate repair entries)
2. Merges coded results with pipeline_input_900.csv
3. Outputs a clean final CSV with all 900 stories

RUN AFTER repair_pipeline() is fully complete.
=============================================================
"""

import pandas as pd, json
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_FILE   = '/content/drive/MyDrive/Microsoft900cases/pipeline_input_900.csv'
CODED_FILE   = '/content/drive/MyDrive/Microsoft900cases/llm_coded_results.jsonl'
OUTPUT_CSV   = '/content/drive/MyDrive/Microsoft900cases/final_coded_900.csv'

# ── STEP 1: Load & deduplicate coded results ──────────────────────────────────
# If a story was coded multiple times, keep the LAST entry (most recent repair)
all_records = {}
total_lines = 0

with open(CODED_FILE, encoding='utf-8') as f:
    for line in f:
        try:
            r = json.loads(line)
            idx = r.get('_meta', {}).get('story_index')
            if idx is not None:
                all_records[idx] = r   # overwrites earlier entries → keeps latest
                total_lines += 1
        except:
            pass

print(f"Total lines in JSONL       : {total_lines}")
print(f"Unique stories after dedup : {len(all_records)}")

# ── STEP 2: Flatten to dataframe ──────────────────────────────────────────────
rows = []
for idx, r in all_records.items():
    meta = r.get('_meta', {})
    rows.append({
        'story_index'                   : idx,
        'validation_issues'             : '; '.join(meta.get('validation_issues', [])) or None,
        'arch_level'                    : r.get('arch_level'),
        'agency_type'                   : r.get('agency_type'),
        'data_connectivity'             : r.get('data_connectivity'),
        'integration_framing'           : r.get('integration_framing'),
        'security_priority_level'       : r.get('security_priority_level'),
        'roi_metric'                    : ', '.join(r.get('roi_metric') or []),
        'roi_numerical'                 : ' | '.join(r.get('roi_numerical') or []) or None,
        'primary_user_persona'          : ', '.join(r.get('primary_user_persona') or []),
        'arch_level_signal_sentences'   : ' | '.join(r.get('arch_level_signal_sentences') or []),
        'agency_evidence_sentence'      : r.get('agency_evidence_sentence'),
        'connectivity_evidence_sentence': r.get('connectivity_evidence_sentence'),
        'roi_evidence_sentences'        : ' | '.join(r.get('roi_evidence_sentences') or []),
        'framing_evidence_sentence'     : r.get('framing_evidence_sentence'),
        'security_evidence_sentence'    : r.get('security_evidence_sentence'),
        'key_quote_text'                : (r.get('key_quote') or {}).get('quote'),
        'key_quote_speaker'             : (r.get('key_quote') or {}).get('speaker'),
        'key_quote_title'               : (r.get('key_quote') or {}).get('title'),
        'technical_summary'             : r.get('technical_summary'),
        'coding_notes'                  : r.get('coding_notes'),
    })

df_coded = pd.DataFrame(rows)

# ── STEP 3: Load input CSV ────────────────────────────────────────────────────
df_input = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')
df_input = (df_input[df_input['Customer Name'].str.strip().notna() &
                     (df_input['Customer Name'].str.strip() != '')]
            .reset_index(drop=True))
df_input.index.name = 'story_index'
df_input = df_input.reset_index()

print(f"Input stories              : {len(df_input)}")

# ── STEP 4: Merge ─────────────────────────────────────────────────────────────
df_merged = df_input.merge(df_coded, on='story_index', how='left')

# ── STEP 5: Report ────────────────────────────────────────────────────────────
total        = len(df_merged)
coded_ok     = df_merged['arch_level'].notna().sum()
missing      = df_merged['arch_level'].isna().sum()

print(f"\n── Merge summary ──────────────────────────────")
print(f"Total rows                 : {total}")
print(f"Successfully coded         : {coded_ok}")
print(f"Missing (not coded)        : {missing}")

if missing > 0:
    print("\n⚠ Stories still missing coding:")
    print(df_merged[df_merged['arch_level'].isna()][['story_index', 'Customer Name']].to_string())

# ── STEP 6: Save ──────────────────────────────────────────────────────────────
df_merged.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"\n✓ Saved to: {OUTPUT_CSV}")
print(f"  Columns  : {len(df_merged.columns)}")
print(f"  Rows     : {len(df_merged)}")

Total lines in JSONL       : 900
Unique stories after dedup : 900
Input stories              : 900

── Merge summary ──────────────────────────────
Total rows                 : 900
Successfully coded         : 900
Missing (not coded)        : 0

✓ Saved to: /content/drive/MyDrive/Microsoft900cases/final_coded_900.csv
  Columns  : 25
  Rows     : 900


In [20]:
# ============================================================
# MERGE BLOCK — cleaned_scraped_900 × final_coded_900
# ============================================================
# Joins LLM-coded variables onto the cleaned scraped dataset.
# Merge key: Customer Name + Products Used (handles duplicate
# company names that appear in multiple stories).
# Result: master_dataset_900 with 33 columns, 900 rows, zero
# nulls on industry_reclassified.
# ============================================================

import pandas as pd

# ── Load ──────────────────────────────────────────────────────────────────────
df_clean = pd.read_csv('/content/drive/MyDrive/Microsoft900cases/cleaned_scraped_900.csv')
df_coded = pd.read_csv('/content/drive/MyDrive/Microsoft900cases/final_coded_900.csv')

# ── Standardise join keys ─────────────────────────────────────────────────────
# cleaned_scraped uses snake_case column names — rename to match coded dataset
df_clean = df_clean.rename(columns={
    'company_name':   'Customer Name',
    'products_used':  'Products Used',
})

# ── Merge ─────────────────────────────────────────────────────────────────────
# Left join: coded dataset is the base (900 rows guaranteed)
# Bring in all contextual columns from cleaned_scraped
df_master = df_coded.merge(
    df_clean[[
        'Customer Name',
        'Products Used',
        'industry_reclassified',   # 13-sector consolidated industry (zero nulls)
        'industry_raw',            # original Microsoft label (for reference)
        'country',
        'org_size',
        'primary_need',
        'triadic_layers_count',    # supply-side triadic breadth (0–3)
        'arch_level_est',          # heuristic embeddedness estimate (1–3)
        'product_stack_breadth',   # total number of products used
    ]],
    on=['Customer Name', 'Products Used'],
    how='left'
)

# ── Validation ────────────────────────────────────────────────────────────────
assert len(df_master) == 900, f"Row count changed: {len(df_master)}"
assert df_master['industry_reclassified'].isna().sum() == 0, "Nulls in industry_reclassified"
assert df_master['arch_level'].isna().sum() == 0, "Nulls in arch_level"

print('✓ Merge successful')
print(f'  Shape: {df_master.shape}')
print(f'  Columns: {df_master.columns.tolist()}')
print()
print('=== arch_level (LLM) ===')
print(df_master['arch_level'].value_counts().sort_index())
print()
print('=== industry_reclassified ===')
print(df_master['industry_reclassified'].value_counts())
print()
print('=== arch_level × industry_reclassified ===')
print(pd.crosstab(df_master['industry_reclassified'], df_master['arch_level']))

# ── Save ──────────────────────────────────────────────────────────────────────
output_path = '/content/drive/MyDrive/Microsoft900cases/master_dataset_900.csv'
df_master.to_csv(output_path, index=False)
print(f'\n✓ Saved to {output_path}')
df_master.head(5)

✓ Merge successful
  Shape: (900, 33)
  Columns: ['story_index', 'Customer Name', 'Products Used', 'Story Content', 'Executive summary', 'validation_issues', 'arch_level', 'agency_type', 'data_connectivity', 'integration_framing', 'security_priority_level', 'roi_metric', 'roi_numerical', 'primary_user_persona', 'arch_level_signal_sentences', 'agency_evidence_sentence', 'connectivity_evidence_sentence', 'roi_evidence_sentences', 'framing_evidence_sentence', 'security_evidence_sentence', 'key_quote_text', 'key_quote_speaker', 'key_quote_title', 'technical_summary', 'coding_notes', 'industry_reclassified', 'industry_raw', 'country', 'org_size', 'primary_need', 'triadic_layers_count', 'arch_level_est', 'product_stack_breadth']

=== arch_level (LLM) ===
arch_level
1    204
2    117
3    579
Name: count, dtype: int64

=== industry_reclassified ===
industry_reclassified
Professional Services         176
Financial Services            117
Manufacturing                  94
Retail & Consumer     

,story_index,Customer Name,Products Used,Story Content,Executive summary,validation_issues,arch_level,agency_type,data_connectivity,integration_framing,...,technical_summary,coding_notes,industry_reclassified,industry_raw,country,org_size,primary_need,triadic_layers_count,arch_level_est,product_stack_breadth
0,0,Intermountain Health,Microsoft Dragon Copilot,Intermountain Health is the largest nonprofit ...,"Intermountain Health, facing some clinician bu...",NaN,3,Tool,System,Strategic,...,Intermountain Health deployed Microsoft Dragon...,Dragon Copilot is a pre-built Microsoft AI pro...,Health,Health Provider,NaN,"10,000+ employees",Employee experience,1,3,1
1,1,Microsoft,Microsoft Copilot Studio\nMicrosoft Foundry,"Every day, millions of people visit Microsoft’...","Microsoft used Copilot Studio to build ""Ask Mi...",NaN,3,Agent,Internal,Tactical,...,Microsoft built the 'Ask Microsoft' web agent ...,This is an ISV/vendor story where Microsoft is...,Technology,"Software, Data and Platforms",NaN,"10,000+ employees",Low-code development,2,3,2
2,2,PwC,Microsoft 365 Enterprise\nMicrosoft 365 Copilo...,"For global firms, standing still is the fastes...","With more than 364,000 people across 130-plus ...",NaN,2,Tool,Internal,Strategic,...,PwC deployed Microsoft 365 with Copilot across...,Copilot Studio is present (Level 2 signal) but...,Professional Services,IT Services and Business Advisory,NaN,"10,000+ employees",Customer experience,1,2,3
3,3,PepsiCo,Microsoft Teams\nMicrosoft 365 Copilot\nMicros...,Even the most agile enterprises face challenge...,"With more than 320,000 employees across 200 co...",NaN,1,Tool,Internal,Tactical,...,PepsiCo deployed Microsoft 365 Copilot across ...,M365 Copilot is a neutral product per codebook...,Retail & Consumer,Consumer Goods,NaN,"10,000+ employees",Artificial Intelligence,1,1,4
4,4,Replit,Azure OpenAI in Foundry Models\nMicrosoft Foun...,Replit’s mission: Empower the next billion sof...,As Replit expanded its agent-driven software c...,NaN,3,Agent,System,Strategic,...,"Replit built an agent-driven, natural-language...",This is an ISV/vendor story: Replit is the pro...,Technology,"Software, Data and Platforms",NaN,50-999 employees,Artificial Intelligence,2,3,6
